# LLM-Powered Stock Trading Agent

## Paper Trading with LLM Decision Making, News Sentiment & RAG Memory

---

**Student Information**

Name: __________

Student ID: __________

> **Update (Oct 11):** Alpha Vantage made news (`NEWS_SENTIMENT`) and `MACD` premium-only. Section 1.9 now gets news from **Alpaca**, and Task 2 computes MACD from **yfinance** prices. RSI and GLOBAL_QUOTE still use Alpha Vantage.

---

### Assignment Overview

In this assignment, you will build components of an **AI-powered stock trading agent** that:
1. Uses real-time financial news and sentiment data to inform decisions
2. Extends data sources with custom retrieval functions (MCP Styled Tools)
3. Learns from past trades using a RAG memory system (BM25 + Reflection)
4. Executes paper trades via the Alpaca brokerage API

### Task Breakdown

| Task | Topic | Status |
|------|-------|--------|
| **Task 1** | Environment, Infrastructure & Core Logic | TODO |
| **Task 2** | MCP Tool Functions (Additional Data Sources) | TODO |
| **Task 3** | RAG Memory System (BM25 + Reflection) | TODO |
| **Task 4** | SFT Fine-tuning with LoRA | TODO |
| **Task 5** | Full Trading Loop Integration | TODO |


---
## Task 1: Environment, Infrastructure & Core Logic

### Grading criteria

- [ ] `llm_agent_decide()` (1.12) runs the multi-turn tool loop and returns 4 values. *Evidence: 1.15 output (rerun after Task 2) shows, turn by turn, the tool called → the first lines of its result → the model's final raw text → a decision that can be executed.*
- [ ] Most days in the 4a backtest have a real decision: the model calls a tool and gives at least one valid buy or sell, rather than the code falling back to a default. *Evidence: 4a output: `Episodes` (days logged) out of the backtest days. Run 4a before 4c, and keep the turn-by-turn log.*
- If your agent can HOLD, explain its HOLD decisions in your report with 2–3 examples from your log.


### 1.1 Install Dependencies

In [ ]:
!pip install -q transformers accelerate alpaca-py rank_bm25 requests peft trl datasets yfinance
# Colab's preinstalled torchao is too old for current peft (Task 4 LoRA fails). It is not used here.
!pip uninstall -y torchao

### 1.2 Imports

In [ ]:
import os
import json
import time
import datetime
import requests
from typing import Dict, Any, List, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

from rank_bm25 import BM25Okapi

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Data
import pandas as pd
import yfinance as yf

# Fine-tuning (Task 5)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

### 1.3 Mount Google Drive

News data will be cached to Google Drive so you don't re-fetch it every run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/stock_agent_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache directory: {CACHE_DIR}")

### 1.4 [TODO] Configuration & API Keys

---

#### Step 1 — Add your API keys

Click the **key icon** in the left sidebar → **Colab Secrets** → add each key:

| Secret name | Where to get it |
|-------------|-----------------|
| `ALPACA_API_KEY` | [alpaca.markets](https://alpaca.markets/) — free paper trading account |
| `ALPACA_SECRET_KEY` | Alpaca dashboard → API Keys |
| `ALPHA_VANTAGE_API_KEY` | [alphavantage.co](https://www.alphavantage.co/support/#api-key) — free tier (25 calls/day) |

---

#### Step 2 — Understand the configurable parameters

The bottom of the code cell below is yours to edit. Here is what each variable does:

**Trading Parameters**

| Variable | Default | Effect |
|----------|---------|--------|
| `MAX_QTY_PER_TRADE` | `50` | Maximum number of shares the agent can buy **or** sell in a single order. Lower this (e.g. `5`) for a conservative strategy; raise it to take larger positions. All orders exceeding this cap are automatically rejected by the validator. |
| `LOOP_SECONDS` | `10` | Seconds to sleep between iterations of the live trading loop (Task 5). Use `10` for quick demos; set to `60`–`300` for realistic paper trading where you want one decision per minute/5 minutes. |
| `ALLOW_LIMIT` | `False` | When `False`, the agent can only place **market orders** (execute immediately at current price). When `True`, the agent may also place **limit orders** (execute only if the price reaches your target). Start with `False` — it keeps the order logic simpler. |

**Memory & Reflection Parameters** *(Task 3 and onward)*

| Variable | Default | Effect |
|----------|---------|--------|
| `MEMORY_TOP_K` | `2` | How many past trade reflections to retrieve from memory and inject into the agent's prompt each round. `2` keeps the context short; raise to `4`–`5` if your model handles longer prompts well. Too high and it crowds out the current market state. |
| `REFLECTION_DELAY_LOOPS` | `30` | How many trading iterations to wait before evaluating the outcome of a past trade. This gives the market time to move before you ask "was that a good trade?" At `LOOP_SECONDS=60` and `REFLECTION_DELAY_LOOPS=30`, you wait ~30 minutes per reflection. Lower it to `5`–`10` for backtesting; keep it at 30+ for live trading. |

---

#### Step 3 — Optionally extend the tradeable universe

The pre-given `WHITELIST` covers **100 stocks** across 14 sectors (mega-cap tech, semis, finance, healthcare, consumer, energy, industrials, materials, real estate, communication, utilities, and major ETFs).

**The agent does NOT fetch news for all 100 at once.** Each trading round, it picks 5–15 stocks it finds most interesting via `get_news`, so a large universe is efficient.

You have three options:

**Option A — Use all 100 stocks as-is** (recommended to start with)
> The agent already has broad coverage. Just run the notebook as provided.

**Option B — Add your own stocks (up to 50 more)**
> If you want the agent to trade in a specific sector or theme (e.g. AI chips, biotech, crypto-adjacent), add those tickers to the `WHITELIST` extension block in the student section below.
>
> Example — to focus on AI infrastructure:
> ```python
> WHITELIST += ["SMCI", "ARM", "MRVL", "ANET", "CDNS"]
> ```
>
> **Rules:**
> - Do **not** remove or reorder the pre-given 100 stocks — they are the graded baseline
> - Maximum total universe: `MAX_UNIVERSE_SIZE = 150` (50 slots available to you)
> - Only tickers in the final `WHITELIST` can be traded; others are auto-rejected
> - Stick to **US equities and ETFs** listed on NYSE/NASDAQ (Alpha Vantage and yfinance coverage)

**Option C — Let the agent discover stocks from news (advanced)**
> Instead of pre-defining target stocks, you implement a **market discovery tool** in Task 2
> that fetches broad market news *without* specifying any ticker symbols. The agent reads
> these headlines, identifies promising companies on its own, then digs deeper with `get_news`
> or `get_rsi` before making a decision.
>
> Use the Alpaca News API from Section 1.9 **without** `symbols`: it returns the latest
> market-wide news, and each article has a `symbols` list with the companies mentioned, so the
> agent gets both the story and the relevant tickers in one call. Alpaca has no topic filter and
> no sentiment scores; to keep a `topic` argument, filter the headlines by keyword yourself.
> (Alpha Vantage's `NEWS_SENTIMENT` is now premium-only.)
>
> **How to wire it up in Task 2:**
> ```python
> def search_market_news(topic: str = "technology") -> str:
>     """Fetch broad market news; returns top mentioned tickers + headlines."""
>     # Call the Alpaca News API without symbols= (latest market-wide news)
>     # Optional: keep only articles whose headline/summary mentions topic
>     # Count each article's symbols; return formatted string the agent can read
>     raise NotImplementedError("TODO")
>
> register_tool(
>     "search_market_news",
>     "Fetch the latest market-wide news (no ticker needed) and the most "
>     "mentioned tickers. Optional topic keyword, e.g. technology, earnings, ipo.",
>     {"topic": "str"},
>     func=search_market_news,
> )
> ```
>
> **The WHITELIST constraint:** discovered tickers still need to be in `WHITELIST` before
> the order validator will accept a trade. Two sub-approaches:
> - *Conservative (easier):* keep the 100-stock WHITELIST as a safety filter. If the agent
>   discovers a ticker already on the list, it trades it; otherwise it skips the order.
>   No extra code needed.
> - *Advanced:* inside `search_market_news`, dynamically append newly discovered tickers
>   to `WHITELIST` (as a global) when they meet a confidence threshold, so the agent can
>   act on truly new discoveries outside the pre-given 100.
>
> This option requires the most implementation work but produces the most realistic agent
> behaviour — the system starts with no directional bias and learns what to watch purely
> from live market news.



In [ ]:
# =============================================================================
# API Keys
# Option 1 (recommended): Add keys via Colab Secrets (lock icon in left sidebar)
#   -> safer: keys are not visible in the notebook file
# Option 2: Paste your keys directly into the strings below
#   -> fine for personal use, but don't share the notebook with keys inside
# =============================================================================

# --- Paste your keys here if NOT using Colab Secrets ---
ALPACA_API_KEY        = "YOUR API KEYS HERE"
ALPACA_SECRET_KEY     = "YOUR API KEYS HERE"
ALPHA_VANTAGE_API_KEY = "YOUR API KEYS HERE"

# --- Colab Secrets override (runs automatically if secrets are set) ---
try:
    from google.colab import userdata
    ALPACA_API_KEY        = userdata.get('ALPACA_API_KEY')        or ALPACA_API_KEY
    ALPACA_SECRET_KEY     = userdata.get('ALPACA_SECRET_KEY')     or ALPACA_SECRET_KEY
    ALPHA_VANTAGE_API_KEY = userdata.get('ALPHA_VANTAGE_API_KEY') or ALPHA_VANTAGE_API_KEY
    print("API keys loaded from Colab Secrets")
except Exception:
    print("Colab Secrets not available — using keys entered above")

os.environ["ALPACA_API_KEY"]   = ALPACA_API_KEY
os.environ["ALPACA_SECRET_KEY"] = ALPACA_SECRET_KEY
os.environ["ALPACA_PAPER"]     = "true"

# =============================================================================
# DO NOT MODIFY — Pre-given Tradeable Universe (100 stocks, 14 sectors)
# ─ The agent's tools (get_news, get_rsi, etc.) query any subset of these.
# ─ Orders for symbols NOT in WHITELIST are automatically rejected.
# ─ Do not remove or reorder these 100 entries.
# =============================================================================
WHITELIST = [
    # Mega-cap Tech (10)
    "AAPL", "MSFT", "NVDA", "TSLA", "GOOG", "AMZN", "META", "NFLX", "ADBE", "ORCL",
    # Semiconductors (8)
    "AMD", "INTC", "AVGO", "QCOM", "MU", "AMAT", "LRCX", "KLAC",
    # Other Tech / Software (8)
    "CRM", "UBER", "PLTR", "SNOW", "CSCO", "IBM", "RBLX", "SNAP",
    # Finance (10)
    "JPM", "BAC", "GS", "MS", "WFC", "V", "MA", "PYPL", "AXP", "BLK",
    # Healthcare (10)
    "JNJ", "UNH", "PFE", "ABBV", "MRK", "LLY", "AMGN", "GILD", "BMY", "CVS",
    # Consumer Staples (6)
    "WMT", "KO", "PEP", "PG", "COST", "MO",
    # Consumer Discretionary (8)
    "MCD", "NKE", "DIS", "SBUX", "TGT", "HD", "F", "GM",
    # Energy (6)
    "XOM", "CVX", "COP", "SLB", "EOG", "MPC",
    # Industrials (8)
    "CAT", "BA", "GE", "HON", "UPS", "FDX", "DE", "RTX",
    # Materials & Commodities (6)
    "LIN", "NEM", "FCX", "DOW", "APD", "NUE",
    # Real Estate (5)
    "AMT", "PLD", "EQIX", "O", "SPG",
    # Communication (5)
    "T", "VZ", "CMCSA", "TMUS", "CHTR",
    # Utilities (1)
    "NEE",
    # Broad ETFs (4)
    "SPY", "QQQ", "IWM", "DIA",
    # Sector ETFs (5)
    "XLF", "XLK", "XLE", "XLV", "XLI",
]

# =============================================================================
# STUDENT CONFIGURATION — Edit the values in this section
# =============================================================================

# ── Universe Extension ────────────────────────────────────────────────────────
# Want to focus on a specific sector or theme? Add your tickers here.
# The agent discovers opportunities via get_news and your MCP tools (Task 2).
# Rules: do NOT remove the 100 above | max total = MAX_UNIVERSE_SIZE = 150
# Example: WHITELIST += ["SMCI", "ARM", "MRVL", "ANET"]
WHITELIST += [
    # Add your symbols here:

]

# ── Trading Parameters ────────────────────────────────────────────────────────
MAX_QTY_PER_TRADE = 50     # Max shares per order. Lower = more conservative.
                            # Valid range: 1–100. Default: 50.

LOOP_SECONDS = 10           # Seconds between live trading iterations (Task 5).
                            # Demo: 10s  |  Realistic paper trading: 60–300s.

ALLOW_LIMIT = False         # False = market orders only (always fills immediately).
                            # True  = agent may also place limit orders at a target price.

# ── Memory & Reflection Parameters ───────────────────────────────────────────
MEMORY_TOP_K = 2            # Past reflections injected into agent context each round.
                            # Higher = richer context. Too high crowds out market data.
                            # Suggested range: 1–5. Default: 2.

REFLECTION_DELAY_LOOPS = 30 # Loops to wait before scoring a past trade's outcome.
                            # At LOOP_SECONDS=60 → delay=30 means ~30 min lookback.
                            # Backtest: use 5–10.  Live trading: use 20–50.

# =============================================================================
# DO NOT MODIFY — Infrastructure constants
# =============================================================================
MAX_UNIVERSE_SIZE      = 150   # Hard cap on WHITELIST length
ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"
NEWS_LIMIT             = 5
MODEL_NAME             = "Qwen/Qwen2.5-1.5B-Instruct"

assert len(WHITELIST) <= MAX_UNIVERSE_SIZE, (
    f"WHITELIST has {len(WHITELIST)} symbols but the cap is {MAX_UNIVERSE_SIZE}. "
    "Remove some symbols from your extension block."
)
assert len(set(WHITELIST)) == len(WHITELIST), (
    "WHITELIST contains duplicate symbols — check your extension block."
)

print("\nConfiguration loaded:")
print(f"  Model:          {MODEL_NAME}")
print(f"  Universe:       {len(WHITELIST)} stocks  (cap: {MAX_UNIVERSE_SIZE})")
print(f"  Max qty/trade:  {MAX_QTY_PER_TRADE} shares")
print(f"  Loop interval:  {LOOP_SECONDS}s")
print(f"  Limit orders:   {ALLOW_LIMIT}")
print(f"  Memory top-k:   {MEMORY_TOP_K}")
print(f"  Reflect delay:  {REFLECTION_DELAY_LOOPS} loops")
print(f"  Paper trading:  True")


### 1.5 Utility Functions

LLMs return **raw text**, not structured data. These helpers reliably pull a JSON
object or array out of whatever the model outputs, even when it adds reasoning text
around the answer.

```
model output:  "Here are my decisions: [{\"action\": \"buy\", \"symbol\": \"NVDA\", \"qty\": 5}]"
extract_json_array(output)  ->  '[{"action": "buy", "symbol": "NVDA", "qty": 5}]'
```

Used internally by the agent loop — you don’t need to call these directly.


In [ ]:
# =============================================================================
# DO NOT MODIFY - JSON Extraction Utilities
# =============================================================================

def extract_last_json(text: str):
    """Extract the last valid JSON object {...} from text."""
    i = text.rfind("{")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "{":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


def extract_json_array(text: str):
    """Extract the first valid JSON array [...] from text."""
    i = text.find("[")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "[":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "]":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


# Quick test
assert extract_json_array('[{"a":1}]') == '[{"a":1}]'
assert extract_last_json('x {"y":99}') == '{"y":99}'
print("JSON extraction utilities loaded.")

### 1.6 Load LLM Model

Loads **Qwen2.5-1.5B-Instruct** on T4 GPU (~30 seconds).


In [ ]:
# =============================================================================
# DO NOT MODIFY - Model Loading
# =============================================================================

def load_model(model_name=MODEL_NAME):
    print(f"Loading model: {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    device = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Model loaded on {device} (dtype={dtype})")
    return tok, model

tok, model = load_model()

### 1.7 Quick Model Test

In [ ]:
test_prompt = "<|system|>\nYou are a helpful assistant.\n<|user|>\nWhat is paper trading in one sentence?\n<|assistant|>\n"
inputs = tok(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False, eos_token_id=tok.eos_token_id)
response = tok.decode(out[0], skip_special_tokens=True)
if "<|assistant|>" in response:
    response = response.split("<|assistant|>")[-1].strip()
print("LLM Response:", response[:300])

### 1.8 Alpaca Trading Client


In [ ]:
# =============================================================================
# DO NOT MODIFY - Trading Infrastructure
# =============================================================================

def trading_client() -> TradingClient:
    key = os.environ["ALPACA_API_KEY"]
    secret = os.environ["ALPACA_SECRET_KEY"]
    paper = os.environ.get("ALPACA_PAPER", "true").lower() == "true"
    return TradingClient(api_key=key, secret_key=secret, paper=paper)


def get_state(tc: TradingClient) -> Dict[str, Any]:
    acct = tc.get_account()
    positions = tc.get_all_positions()
    pos = []
    for p in positions:
        pos.append({
            "symbol": p.symbol,
            "qty": float(p.qty),
            "market_value": float(p.market_value),
            "avg_entry_price": float(p.avg_entry_price) if p.avg_entry_price is not None else None,
        })
    return {
        "cash": float(acct.cash),
        "equity": float(acct.equity),
        "buying_power": float(acct.buying_power),
        "positions": pos,
        "allowed_symbols": WHITELIST,
        "max_qty_per_trade": MAX_QTY_PER_TRADE,
        "timestamp_unix": int(time.time()),
    }


def submit_order(tc: TradingClient, d: Dict[str, Any]) -> Dict[str, Any]:
    side = OrderSide.BUY if d["action"] == "buy" else OrderSide.SELL
    if d["order_type"] == "limit":
        req = LimitOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY, limit_price=str(d["limit_price"]),
        )
    else:
        req = MarketOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY,
        )
    order = tc.submit_order(order_data=req)
    return {
        "id": str(order.id), "status": str(order.status),
        "symbol": d["symbol"], "side": d["action"], "qty": d["qty"],
    }


# --- Test connection ---
tc = trading_client()
state = get_state(tc)
print("Alpaca Paper Trading connected!")
print(f"  Equity:       ${state['equity']:,.2f}")
print(f"  Cash:         ${state['cash']:,.2f}")
print(f"  Buying Power: ${state['buying_power']:,.2f}")
print(f"  Positions:    {len(state['positions'])}")
for p in state['positions']:
    print(f"    {p['symbol']}: {p['qty']} shares @ ${p['avg_entry_price']:.2f}")

### 1.9 [TODO] News Retrieval (Google Drive Cached)

`get_news_for_symbols()` fetches financial news from the **Alpaca News API** (same keys as your
paper trading account) and caches results to Google Drive by date.

> **Update (Oct 11):** this section used Alpha Vantage `NEWS_SENTIMENT`, which is now premium-only.
> If you already implemented it with Alpha Vantage, only the API call and the parsing change.
> Delete any `news_*.json` in `CACHE_DIR` that says "No news data available".

**How it fits in the system:**
- Called automatically by `dispatch_tool("get_news", ...)` when the trading agent
  requests news during its decision loop
- The backtest does not pre-fetch news: the agent's first `get_news` call for a date fetches
  from the API and is cached; later calls for that date read the cache
- Live trading fetches on demand and caches for the rest of the day

**Caching strategy:**
- Cache file: `{CACHE_DIR}/news_{YYYY-MM-DD}.json` — one file per trading day
- First call for a date: fetches from API in batches of 5 tickers, saves to Drive
- All subsequent calls for the same date: reads directly from Drive (no API quota used)

**Alpaca news note:**
The free plan allows 200 calls per minute and has no daily limit, so the 4a backtest can fill
the cache in one run. `limit` counts articles for the whole batch, so one busy ticker can fill
it; smaller batches (even one ticker per call) are fine. Some stocks have no news on some days:
an empty list is a valid answer. There are no sentiment scores; the model reads the headlines
and summaries.


In [ ]:
# =============================================================================
# News Retrieval with Google Drive Caching
# Agent get_news calls fetch on demand; results are cached by date.
# =============================================================================

def _get_news_cache_path(date_str=None):
    """Return the Drive cache file path for a given date."""
    if date_str is None:
        date_str = datetime.date.today().isoformat()
    return os.path.join(CACHE_DIR, f"news_{date_str}.json")


def get_news_for_symbols(symbols: list, limit: int = NEWS_LIMIT,
                         date: str = None) -> str:
    """
    Fetch financial news for the given symbols, with Google Drive caching.

    Called by dispatch_tool() when the agent requests get_news.
    In backtest mode, the agent's get_news calls pass the backtest date, so the
    cache is per historical day. (run_backtest() does not pre-fetch news.)

    Args:
        symbols : list of ticker strings, e.g. ["AAPL", "NVDA"]
        limit   : max articles per API batch (default NEWS_LIMIT)
        date    : "YYYY-MM-DD" for historical queries; None = today

    Returns:
        Formatted string with headlines, sources, tickers, and summaries.
        Returns "No news data available" if the API is unreachable.

    ============================================================================
    TODO: Implement get_news_for_symbols()
    ============================================================================

    This function demonstrates the complete pattern for implementing API-based
    tools: API calls → error handling → caching → formatting. Implement the
    other functions (get_price_history, get_rsi, get_macd, get_global_quote)
    following this same pattern.

    IMPLEMENTATION STEPS:
    ---------------------

    STEP 1: Parse Date Parameter
    -----------------------------
    - If date is None, use today's date (datetime.date.today().isoformat())
    - Store as date_str for consistent formatting
    - Call _get_news_cache_path(date_str) to get the cache file path

    STEP 2: Check Cache (Read from Google Drive)
    ---------------------------------------------
    - Use os.path.exists() to check if cache file exists
    - If exists:
        * Open with json.load() and read the cached data
        * Print a cache hit message: f"  [News] Cache hit ({date_str})"
        * Return the cached["text"] value immediately
        * Wrap in try/except to handle corrupted cache files
    - If cache read fails, continue to API fetch (don't return error)

    STEP 3: Fetch from the Alpaca News API (Batch Processing)
    ---------------------------------------------------------
    Alpaca News API Details (free plan: 200 calls/min, no daily limit):
    - Endpoint: "https://data.alpaca.markets/v1beta1/news"
    - Headers: "APCA-API-KEY-ID": ALPACA_API_KEY,
               "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY
    - Parameters:
        * symbols: comma-separated string (e.g., "AAPL,NVDA,TSLA")
        * limit: max articles per request, for the whole batch (use NEWS_LIMIT; max 50)
        * start: (optional) "YYYY-MM-DDTHH:MM:SSZ" for historical queries
        * end: (optional) "YYYY-MM-DDTHH:MM:SSZ" for historical queries

    Historical Query Setup:
    - If date parameter is provided (not None):
        * Set start = f"{date_str}T00:00:00Z" (start of day)
        * Set end = f"{date_str}T23:59:59Z" (end of day)

    Batch Processing:
    - Split symbols list into batches of 5
    - Use: batches = [symbols[i:i+5] for i in range(0, len(symbols), 5)]
    - Process each batch separately with time.sleep(2) between batches
    - limit counts articles for the whole batch, so one busy ticker can fill it;
      smaller batches (even 1 ticker) are fine

    API Call Pattern:
    ```python
    resp = requests.get("https://data.alpaca.markets/v1beta1/news", params={
        "symbols": ",".join(batch),
        "limit": limit,
        # add start and end if date is provided
    }, headers={
        "APCA-API-KEY-ID": ALPACA_API_KEY,
        "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY,
    }, timeout=30)
    data = resp.json()
    ```

    Error Handling:
    - If resp.status_code != 200 → print the status code, continue to next batch
      (do not cache that result)
    - An empty data["news"] list is a valid answer (some stocks have no news on
      some days): format it as 0 articles
    - Wrap each batch in try/except to handle network errors
    - If all batches fail, return "No news data available"

    STEP 4: Parse Response and Format Output
    -----------------------------------------
    Response Structure:
    ```json
    {
      "news": [
        {
          "headline": "Article headline",
          "source": "benzinga",
          "created_at": "2025-01-15T15:00:33Z",
          "summary": "Article summary text...",
          "symbols": ["AAPL", "MSFT"],
          "url": "https://..."
        }
      ],
      "next_page_token": null
    }
    ```
    There are no sentiment scores. Headlines and summaries can contain HTML
    entities such as "&amp;": use html.unescape() (import html). The summary
    can be empty.

    Formatting Requirements:
    For each batch, create a formatted block:
    - Header: "=== News: {tickers} [{date_str}] ==="
    - Count: "({len(news)} articles)"
    - For each article:
        * [1] Headline
        * Source | created_at
        * Tickers: the article's symbols, comma-separated
        * Summary: first 200 characters...
        * Separator: "-" * 50

    Example Output:
    ```
    === News: AAPL,NVDA [2025-01-15] ===
    (2 articles)

    [1] Apple Announces New Product Line
        benzinga | 2025-01-15T14:30:00Z
        Tickers: AAPL
        Summary: Apple Inc. unveiled its latest innovation...
    --------------------------------------------------

    [2] Tech Sector Analysis
        benzinga | 2025-01-15T12:00:00Z
        Tickers: AAPL, NVDA
        Summary: Industry analysts predict continued growth...
    --------------------------------------------------
    ```

    STEP 5: Save to Cache (Write to Google Drive)
    ----------------------------------------------
    Cache Structure:
    ```python
    {
        "text": formatted_news_string,
        "timestamp": time.time(),
        "date": date_str,
        "symbols": symbols
    }
    ```

    Save Process:
    - Use cache_path from _get_news_cache_path()
    - Write with json.dump(data, f, ensure_ascii=False)
    - Use encoding="utf-8" for file open
    - Wrap in try/except to handle write errors
    - Print error if cache write fails, but still return the text

    STEP 6: Return Formatted String
    --------------------------------
    - Combine all batch results with "\n".join(all_news)
    - If no news found in any batch, return "No news data available"
    - Print success message at the end: "News retrieval loaded."

    ============================================================================
    TESTING YOUR IMPLEMENTATION:
    ============================================================================

    After implementing, test with:
    ```python
    # Test 1: Single symbol, today's news
    result = get_news_for_symbols(["AAPL"])
    print(result)

    # Test 2: Multiple symbols (triggers batching)
    result = get_news_for_symbols(["AAPL", "NVDA", "TSLA", "MSFT", "GOOGL", "META"])
    print(result)

    # Test 3: Historical query
    result = get_news_for_symbols(["AAPL"], date="2025-01-15")
    print(result)

    # Test 4: Cache hit (run same query twice)
    result = get_news_for_symbols(["AAPL"])  # Should print "Cache hit"
    ```

    ============================================================================
    CACHING STRATEGY EXPLAINED:
    ============================================================================

    Why Cache by Date?
    - News for a given date doesn't change after that day ends
    - In backtesting, we query the same historical dates repeatedly
    - Caching saves API calls and makes backtest re-runs fast

    Cache File Naming:
    - Format: {CACHE_DIR}/news_YYYY-MM-DD.json
    - Example: /content/drive/MyDrive/stock_agent_cache/news_2025-01-15.json
    - One file per trading day

    In Backtest:
    - run_backtest() does not call this function itself; the agent's get_news calls do
    - Later calls for the same date read the cache (no API call)

    ============================================================================
    """
    raise NotImplementedError("TODO: implement get_news_for_symbols()")



print("News retrieval loaded.")


### Test 1.9

In [ ]:
# Test get_news_for_symbols implementation
print("Testing 1.9: News Retrieval with Google Drive Caching")
print("=" * 60)

# --- Test 1: Single symbol (today's news) ---
try:
    result = get_news_for_symbols(["AAPL"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 1 PASSED - Single symbol")
    print(f"  Length: {len(result)} chars")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 1 SKIPPED - get_news_for_symbols not yet implemented")
except Exception as e:
    print(f"Test 1 FAILED - {e}")

# --- Test 2: Multiple symbols (triggers batching if >5) ---
try:
    result = get_news_for_symbols(["AAPL", "NVDA", "TSLA", "MSFT", "GOOGL", "META"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 2 PASSED - Multi-symbol batching")
    print(f"  Length: {len(result)} chars")
except NotImplementedError:
    print("Test 2 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 2 FAILED - {e}")

# --- Test 3: Cache hit (re-fetch same date should hit cache) ---
try:
    result2 = get_news_for_symbols(["AAPL"])  # Should print "Cache hit"
    assert isinstance(result2, str), f"Expected str, got {type(result2)}"
    print("Test 3 PASSED - Cache hit on repeated query")
except NotImplementedError:
    print("Test 3 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 3 FAILED - {e}")

# --- Test 4: Historical date query ---
try:
    result = get_news_for_symbols(["AAPL"], date="2025-01-15")
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    print("Test 4 PASSED - Historical date query")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 4 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 4 FAILED - {e}")

# --- Test 5: Cache file structure validation ---
try:
    cache_path = _get_news_cache_path()
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        assert "text" in cached, "Cache missing 'text' key"
        assert "timestamp" in cached, "Cache missing 'timestamp' key"
        assert "date" in cached, "Cache missing 'date' key"
        assert "symbols" in cached, "Cache missing 'symbols' key"
        print("Test 5 PASSED - Cache file structure is correct")
        print(f"  Keys: {list(cached.keys())}")
    else:
        print("Test 5 SKIPPED - No cache file found (API may be unavailable)")
except NotImplementedError:
    print("Test 5 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 5 FAILED - {e}")

print("" + "=" * 60)
print("1.9 Testing complete.")

### 1.10 Tool Registry & Dispatcher

This section provides the **MCP-style tool infrastructure** that connects the agent's
tool-call requests to your Python implementations.

**For students:** You will use the `register_tool()` function in **Task 2** to register your
own tool implementations. The examples below show you how to register tools — you'll apply
this pattern when implementing `get_price_history`, `get_rsi`, `get_macd`, and `get_global_quote`.

#### How it works

```
Agent outputs:  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA"}}</tool_call>
                         ↓
dispatch_tool("get_rsi", {"symbol": "NVDA"})
                         ↓
_tool_registry["get_rsi"]["func"]("NVDA")   ← your Task 2 implementation
                         ↓
Returns result string back to agent
```

#### Tool Registry API

| Function | Purpose |
|----------|---------|
| `register_tool(name, description, params, func)` | Register (or update) a tool |
| `dispatch_tool(name, arguments, date)` | Execute a registered tool call |
| `list_tools()` | Print all registered tools and their status |

#### Adding your own tool (Task 2 Optional)

```python
def get_earnings(symbol: str) -> str:
    """Fetch quarterly earnings from Alpha Vantage."""
    # ... your implementation ...
    return formatted_string

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# The agent can now call get_earnings automatically!
```

#### Pre-registered tools

`get_news` is registered here with the full implementation from 1.9.
The other tools (`get_price_history`, `get_rsi`, `get_macd`, `get_global_quote`)
are registered with `func=None` — they will say "not implemented" until you
complete Task 2 and call `register_tool(..., func=your_function)` to update them.


In [ ]:
# =============================================================================
# Tool Registry & Dispatcher
# =============================================================================
import re as _re

_tool_registry: dict = {}   # name -> {description, params, func}


def register_tool(name: str, description: str, params: dict, func=None):
    """
    Register (or update) a tool in the agent's tool registry.

    Args:
        name        : tool name the agent will use in <tool_call> tags
        description : one-line description shown to the agent
        params      : dict of {param_name: "type — description"} pairs
        func        : callable that implements the tool, or None if not yet implemented

    Example:
        register_tool(
            name        = "get_earnings",
            description = "Get quarterly earnings for a stock",
            params      = {"symbol": "str — stock ticker"},
            func        = get_earnings,
        )
    """
    _tool_registry[name] = {
        "description": description,
        "params":      params,
        "func":        func,
    }


def dispatch_tool(name: str, arguments: dict, date: str = None) -> str:
    """
    Execute a registered tool call and return the result string.

    Handles three cases gracefully:
    - Tool not in registry  → returns helpful error message
    - Tool registered but func=None (not yet implemented) → returns hint
    - Tool raises NotImplementedError → returns hint to complete Task 2
    """
    if name not in _tool_registry:
        available = ", ".join(_tool_registry.keys())
        return f"Unknown tool '{name}'. Available: {available}"

    entry = _tool_registry[name]
    func  = entry.get("func")

    if func is None:
        return (f"[Tool '{name}' is not yet implemented. "
                f"Complete Task 2 and call register_tool('{name}', ..., func=your_fn).]")
    try:
        if name == "get_news":
            # news tool needs the date param for historical backtest
            return func(arguments.get("symbols", []), date=date)
        else:
            return func(**arguments)

    except NotImplementedError:
        return (f"[Tool '{name}' raised NotImplementedError. "
                f"Implement it in Task 2 and re-register.]")
    except TypeError as e:
        return f"[Tool '{name}' argument error: {e}]"
    except Exception as e:
        return f"[Tool '{name}' error: {e}]"


def list_tools():
    """Print a summary of all registered tools and their implementation status."""
    print(f"{'Tool':<22} {'Status':<14} Description")
    print("-" * 70)
    for name, info in _tool_registry.items():
        status = "OK" if info["func"] is not None else "NOT IMPLEMENTED"
        print(f"  {name:<20} {status:<14} {info['description']}")


def _parse_tool_call(text: str):
    """Extract and parse JSON from a <tool_call>...</tool_call> block."""
    match = _re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, _re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    return None


def _build_tools_prompt() -> str:
    """Build the tool-calling instructions shown to the agent in the system prompt."""
    implemented = {k: v for k, v in _tool_registry.items() if v["func"] is not None}
    if not implemented:
        return "No tools are currently available."

    lines = [
        "You can call tools to gather market data. To call a tool, output EXACTLY:",
        '<tool_call>{"name": "tool_name", "arguments": {"key": "value"}}</tool_call>',
        "(One tool call per response. Output NOTHING else when calling a tool.)",
        "",
        "Available tools:",
    ]
    for name, info in implemented.items():
        params_str = ", ".join(f"{k}: {v}" for k, v in info["params"].items())
        lines.append(f"  - {name}({params_str})")
        lines.append(f"      {info['description']}")

    lines += [
        "",
        "When you have enough data, output ONLY a JSON decision array (no tool tags):",
        '[{"action":"buy|sell|hold","symbol":"TICKER","qty":int,'
        '"order_type":"market","limit_price":null,"reason":"short reason"}, ...]',
    ]
    return "\n".join(lines)


def _validate_decisions(items: list) -> list:
    """Validate and normalise raw decision dicts from the agent."""
    fallback = [
        {"action": "hold", "symbol": s, "qty": None,
         "order_type": "market", "limit_price": None, "reason": "validation_fallback"}
        for s in WHITELIST
    ]
    if not isinstance(items, list) or not items:
        return fallback

    out = []
    for obj in items:
        if not isinstance(obj, dict):
            continue
        action = obj.get("action", "hold")
        symbol = obj.get("symbol") or obj.get("ticker")
        qty    = obj.get("qty")
        reason = obj.get("reason", "")

        if action not in ("buy", "sell", "hold"):
            action = "hold"

        if action == "hold":
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None, "reason": reason})
            continue

        if symbol not in WHITELIST:
            continue
        if not isinstance(qty, int) or not (1 <= qty <= MAX_QTY_PER_TRADE):
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None,
                        "reason": "invalid_qty"})
            continue

        out.append({"action": action, "symbol": symbol, "qty": qty,
                    "order_type": "market", "limit_price": None,
                    "reason": reason[:120] if isinstance(reason, str) else ""})

    return out if out else fallback


# =============================================================================
# Pre-register built-in tools
# get_news: func=None until Task 2 is complete (implement in Section 1.9)
# Others:   func=None until Task 2 is complete
# =============================================================================

# Note: get_news implementation is in Section 1.9 (Task 2)
# After implementing get_news_for_symbols(), it will be automatically available here
register_tool(
    name        = "get_news",
    description = (
        "Get financial news + sentiment for specific stocks. "
        "Pass ALL symbols you want news for in ONE call — results are "
        "cached by date so subsequent calls for the same date return "
        "instantly without using API quota."
    ),
    params      = {"symbols": 'list[str] — all tickers you want to investigate, e.g. ["AAPL", "NVDA", "MSFT"]'},
    func        = get_news_for_symbols,  # implement in Section 1.9
)
register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_rsi",
    description = "Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params      = {"symbol": "str", "time_period": "int (default 14)"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_macd",
    description = "Get MACD trend indicator via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_global_quote",
    description = "Get real-time price quote via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)

print("Tool registry loaded.")
list_tools()


### 1.11 [Optional] Find WHITELIST
### What find_whitelist() does:
You can implement a function to search for new tickers to trade and add them into the trading Whitelist. It is up to you to load a small model or use some customized strategies. You are also encouraged to
explore the possibility of using MCP tools instead of externally implement this function.

In [ ]:
def find_whitelist():
  """
    Returns a list of tickers
  """
  tickers = []
  return tickers

### 1.12 [TODO] LLM Agent Decision Engine

<!-- > **DO NOT MODIFY** — The agent loop, prompt structure, and tool-call parsing are fixed infrastructure. -->

#### What `llm_agent_decide()` does

Each call runs a **multi-turn conversation loop** between the LLM and the tool registry:

```
Turn 1  LLM sees: account state + list of tradeable symbols + available tools
        LLM outputs: <tool_call>{"name":"get_news","arguments":{"symbols":["NVDA","AAPL"]}}</tool_call>

Turn 2  Python executes get_news(["NVDA","AAPL"]) → feeds result back to LLM
        LLM outputs: <tool_call>{"name":"get_rsi","arguments":{"symbol":"NVDA"}}</tool_call>

Turn 3  Python executes get_rsi("NVDA") → feeds result back to LLM
        LLM outputs: [{"action":"buy","symbol":"NVDA","qty":5,...}, ...]
                       ↑ Final JSON — loop ends
```

The loop runs for up to `max_turns` iterations. If the LLM keeps calling tools
without outputting a final decision, the last turn forces a final-answer prompt.

The helpers from 1.5 and 1.10 (`_parse_tool_call`, `extract_json_array`, `extract_last_json`, `_validate_decisions`) are a starting point: use them as they are, or write more robust versions in this cell. Describe any change in your report.

#### What it returns

```python
decisions, sft_prompt, sft_completion, n_tool_calls = llm_agent_decide(tok, model, state)
```

| Return value | Type | Description |
|---|---|---|
| `decisions` | `list[dict]` | Validated trading decisions |
| `sft_prompt` | `str` | Full conversation (used as SFT training prompt) |
| `sft_completion` | `str` | Final JSON string (used as SFT training completion) |
| `n_tool_calls` | `int` | Number of tool calls made (`TradingDataCollector.log()` uses it to filter episodes) |

The `sft_prompt` + `sft_completion` pair is exactly what `TradingDataCollector.log()` expects.

#### Effect of Task 2 on the agent

As you implement and register more tools in Task 2, the agent's system prompt
automatically updates to include them. The agent will start using your tools
without any changes to this cell.


In [ ]:
# =============================================================================
# LLM Agent Decision Engine  —  STUDENT IMPLEMENTATION (REQUIRED)
# =============================================================================

def llm_agent_decide(tok, model, state: dict,
                     universe: list = None,
                     date: str = None,
                     max_turns: int = 6):
    """
    MCP-style agent loop: LLM calls tools to research stocks, then outputs decisions.

    TODO: Implement the multi-turn agent loop that:
    1. Builds a system prompt with tool instructions (_build_tools_prompt())
    2. Initializes conversation with account state
    3. Runs up to max_turns iterations:
       a. Generate LLM response using the tokenizer and model
       b. Check if response contains a tool call (_parse_tool_call())
       c. If tool call: execute it (dispatch_tool()), append result to conversation
       d. If no tool call: try to extract final JSON decision array
       e. If valid decisions found: validate and return
       f. If max turns reached: return fallback
    4. Returns (decisions, sft_prompt, sft_completion, n_tool_calls)

    Args:
        tok       : HuggingFace tokenizer
        model     : HuggingFace model (base or LoRA fine-tuned)
        state     : Account state dict from get_state() / SimulatedPortfolio.get_state()
        universe  : Tradeable tickers (defaults to WHITELIST)
        date      : "YYYY-MM-DD" for backtest historical context; None = live trading
        max_turns : Maximum agent iterations before forcing a final decision

    Returns:
        decisions      (list[dict]) — validated trading decision dicts
        sft_prompt     (str)        — full conversation string for SFT training
        sft_completion (str)        — final JSON string for SFT training
        n_tool_calls   (int)        — number of tool calls made in this episode

    Implementation Steps:

    STEP 1: Setup
    - Set universe to WHITELIST if None
    - Create fallback decisions (all "hold" for each symbol in WHITELIST)

    STEP 2: Build System Prompt
    - Call _build_tools_prompt() to get available tools description
    - Create system message with:
      * Trading agent role description
      * Tradeable universe list
      * Tool calling instructions
      * Trading strategy guidelines
      * Trading rules (qty limits, active trading, etc.)

    STEP 3: Initialize Conversation
    - Convert state dict to JSON string
    - Create initial user message with account state
    - Format conversation as: "<|system|>\n{system}\n<|user|>\n{user_msg}\n<|assistant|>\n"
    - Initialize tool_calls_made = []

    STEP 4: Agent Loop (for turn in range(max_turns))

      4a. Generate LLM Response:
          - Tokenize conversation with tok(conversation, return_tensors="pt")
          - Move to GPU if available (torch.cuda.is_available())
          - Call model.generate() with:
            * max_new_tokens=700
            * do_sample=True, temperature=0.7, top_p=0.9
            * eos_token_id=tok.eos_token_id
          - Decode new tokens only (skip input prompt)
          - Strip and store as response

      4b. Parse Response:
          - Try _parse_tool_call(response) to extract tool call JSON

      4c. Handle Tool Call (if tc and turn < max_turns - 1):
          - Extract tool name and arguments
          - Print debug info: f"  [Agent turn {turn+1}] → {name}({list(args.values())})"
          - Execute: result = dispatch_tool(name, args, date=date)
          - Print the first lines of the result: print(f"    {result[:200]}")
          - Track: tool_calls_made.append({"tool": name, "args": args})
          - Append to conversation:
            "{response}\n<|user|>\n[Result of {name}]:\n{result[:3000]}\n<|assistant|>\n"
          - Continue to next turn

      4d. Handle Final Decision (else):
          - Print: f"  [Agent] Finalising — {turn+1} turn(s), {len(tool_calls_made)} tool call(s)"
          - Print the model's final raw text: print(f"    {response[:300]}")
          - Try extract_json_array(response)
          - If None, try extract_last_json(response)
          - If valid JSON found:
            * Parse with json.loads()
            * Convert to list if single dict
            * Validate with _validate_decisions(items)
            * Return (decisions, conversation, json.dumps(decisions), len(tool_calls_made))
          - If no valid JSON and turn < max_turns - 1:
            * Append nudge: "{response}\n<|user|>\nNow output ONLY the final JSON array...\n<|assistant|>\n"
          - If turn >= max_turns - 1: break

    STEP 5: Fallback Return
    - If loop completes without valid decisions:
    - Return (fallback, conversation, json.dumps(fallback), len(tool_calls_made))

    Helper functions you can use:
    - _build_tools_prompt() : str — generates tool description for system prompt
    - _parse_tool_call(text: str) : dict | None — extracts tool call JSON
    - _validate_decisions(items: list) : list — validates and normalizes decisions
    - dispatch_tool(name, args, date) : str — executes tool and returns result
    - extract_json_array(text) : str | None — extracts JSON array from text
    - extract_last_json(text) : str | None — extracts last JSON object from text

    Constants available:
    - WHITELIST : list[str] — all tradeable symbols
    - MAX_QTY_PER_TRADE : int — maximum shares per order
    """
    if universe is None:
        universe = WHITELIST

    fallback = [{"symbol": s, "action": "hold", "qty": 0} for s in universe]

    # ---------- SYSTEM ----------
    tools_desc = _build_tools_prompt()

    system = f"""You are an autonomous portfolio manager.

Your objective is to grow portfolio equity through active trading decisions.

Tradeable universe:
{universe}

==============================
MANDATES
==============================

You MUST actively manage capital.

You are NOT allowed to output only HOLD decisions.

You MUST execute at least one BUY or SELL action in your final decision.

Remaining inactive is considered suboptimal behavior.

If signals are weak, you must still make the best risk-aware trade.

You MUST call at least one tool before making a final decision.

Your FIRST response MUST be a <tool_call>.

Do NOT output trading decisions before calling a tool.

==============================
TOOL USAGE
==============================

To call a tool, respond EXACTLY like this:

<tool_call>
{{"name": "tool_name", "arguments": {{"param": "value"}}}}
</tool_call>

Example:

<tool_call>
{{"name": "get_global_quote", "arguments": {{"symbol": "AAPL"}}}}
</tool_call>

Available tools:
{tools_desc}

==============================
TRADING EXPECTATION
==============================

You are expected to:

- Deploy capital
- Adjust positions
- Take calculated risks
- Avoid passivity

You MUST make at least one active trade.

==============================
FINAL DECISION FORMAT
==============================

After completing your research using tools,
you MUST output your final trading decisions in the following JSON format.

Output ONLY valid JSON.

Do NOT include:
- explanations
- markdown
- commentary
- extra text

The output must be a JSON array.

Each decision must follow this schema:

[
  {{
    "symbol": "TICKER",
    "action": "buy" | "sell",
    "qty": integer
  }}
]

Rules:
- HOLD is NOT allowed
- qty must be > 0
- Use only symbols from the tradeable universe
- At least ONE BUY or SELL must appear

Example:

[
  {{"symbol": "AAPL", "action": "buy", "qty": 5}},
  {{"symbol": "MSFT", "action": "sell", "qty": 3}}
]

A final decision without using tool information is invalid."""
    user_msg = json.dumps(state, indent=2)

    conversation = (
        f"<|system|>\n{system}\n"
        f"<|user|>\nAccount State:\n{user_msg}\n"
        f"<|assistant|>\n"
    )

    tool_calls_made = []



    # To-Do: Finish the LLM decision loop
    # ---------- LOOP ----------
    for turn in range(max_turns):

        inputs = tok(conversation, return_tensors="pt")


    return fallback, conversation, json.dumps(fallback), len(tool_calls_made)


print("LLM agent decision engine loaded (TODO stub - implement in Task 1.12).")


### [Optional to Edit] 1.13 Training Data Utilities

> **Run this cell now, in order.** It only defines tools here; Task 4 uses them (4a collects training data, 4c compares the model before and after training).

`TradingDataCollector` records every agent episode to `sft_data.jsonl` on Google Drive.
Each line in the file is a JSON object containing:

```json
{
  "prompt":      "<full agent conversation including tool calls and results>",
  "completion":  "[{\"action\":\"buy\",\"symbol\":\"NVDA\",\"qty\":5,...}]",
  "equity":      100000.0,
  "equity_after": 101200.0,
  "pnl":         1200.0,
  "label":       "good",
  "timestamp":   1705312800
}
```

**Why store the full conversation as `prompt`?**
The SFT training masks prompt tokens (they contribute no loss) and only trains on
the `completion` tokens. Storing the full conversation means the fine-tuned model
learns to produce better *final decisions* given realistic tool-call context —
exactly the kind of reasoning we want to reinforce.

**Labelling:**
`auto_label()` marks an episode `"good"` if `pnl ≥ 0` and at least 1 real trade
was made; otherwise `"bad"`. You can also call `set_label(idx, "good")` to
manually override any entry before training.

**Workflow:**
```
run_backtest()           → fills sft_data.jsonl with unlabelled episodes
collector.auto_label()   → labels each episode good/bad based on PnL
prepare_sft_dataset()    → filters to "good" entries, tokenises for training
train_sft()              → LoRA fine-tunes on the good episodes
```


In [ ]:
# =============================================================================
# Training Data Collection Utilities
# =============================================================================

class TradingDataCollector:
    """
    Logs agent episodes (conversation + decisions + outcomes) to Google Drive
    for SFT fine-tuning.

    Each episode stored in sft_data.jsonl as:
        prompt      = full agent conversation (system + tool calls + results)
        completion  = final JSON decision string
        equity      = portfolio value before this decision
        equity_after= portfolio value after trading
        pnl         = equity_after - equity
        label       = "good" | "bad" | None (set by auto_label or set_label)
    """

    def __init__(self):
        self.save_path = os.path.join(CACHE_DIR, "sft_data.jsonl")
        self.entries   = []
        if os.path.exists(self.save_path):
            with open(self.save_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.entries.append(json.loads(line))
            print(f"Loaded {len(self.entries)} existing training entries")

    # ── Logging ────────────────────────────────────────────────────────────

    def log(self, sft_prompt: str, sft_completion: str,
            decisions: list, equity: float,
            tool_calls: int = 0) -> int | None:
        """
        Record one agent episode for SFT fine-tuning.

        Hard filters (drop the episode if any is true):
        - No tool was called (tool_calls <= 0)
        - No real trade (no BUY/SELL with qty > 0)
        - Any BUY/SELL decision is a validation fallback (reason == "validation_fallback")
        - Any BUY/SELL has invalid qty (missing / non-positive / non-int)
        """

        # --- Filter 1: must have used at least one tool ---
        # We only log decisions informed by external data retrieval.
        if tool_calls <= 0:
            return None

        # --- Filter 2: must contain at least one real trade ---
        # Keep only BUY/SELL actions (HOLDs do not contribute to learning trading behavior).
        real_trades = [d for d in decisions if isinstance(d, dict) and d.get("action") in ("buy", "sell")]
        if not real_trades:
            return None

        # --- Filter 3: drop any episode that contains validation fallback trades ---
        # We do not want the model to learn from forced/invalid outputs.
        if any(d.get("reason") == "validation_fallback" for d in real_trades):
            return None

        # --- Filter 4: qty must be a positive integer for every real trade ---
        # Prevent logging malformed trades that would be rejected by the executor/validator.
        cleaned_real_trades = []
        for d in real_trades:
            qty = d.get("qty")
            if not isinstance(qty, int) or qty <= 0:
                return None
            cleaned_real_trades.append(d)

        # --- Log valid episode ---
        entry = {
            "prompt":       sft_prompt,
            "completion":   sft_completion,
            "equity":       equity,
            "equity_after": None,
            "pnl":          None,
            "label":        None,
            "timestamp":    int(time.time()),
        }

        self.entries.append(entry)

        with open(self.save_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        return len(self.entries) - 1



    # ── Labelling ──────────────────────────────────────────────────────────

    def auto_label(self, threshold: float = 0.0, min_trades: int = 1):
        """
        Auto-label all entries that have a pnl outcome.

        Label = "good"  if pnl >= threshold AND n_actual_trades >= min_trades
        Label = "bad"   otherwise (passive episode or lost money)
        """
        labeled = 0
        for e in self.entries:
            if e.get("pnl") is not None and e["label"] is None:
                decs     = json.loads(e["completion"])
                n_trades = sum(1 for d in decs if d.get("action") in ("buy", "sell"))
                if n_trades < min_trades:
                    e["label"] = "bad"
                else:
                    e["label"] = "good" if e["pnl"] >= threshold else "bad"
                labeled += 1
        self._save_all()
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Labelled {labeled} episodes  →  {good} good, {bad} bad")

    def set_label(self, idx: int, label: str):
        """Manually override the label of a specific entry."""
        if 0 <= idx < len(self.entries):
            self.entries[idx]["label"] = label
            self._save_all()

    def get_labeled(self, label: str = "good") -> list:
        """Return all entries with the given label."""
        return [e for e in self.entries if e.get("label") == label]

    # ── Review ─────────────────────────────────────────────────────────────

    def summary(self):
        total        = len(self.entries)
        with_outcome = sum(1 for e in self.entries if e.get("pnl") is not None)
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Episodes: {total} | With outcome: {with_outcome} | "
              f"Good: {good} | Bad: {bad} | Unlabelled: {total-good-bad}")

    def review(self, n: int = 10):
        """Print a summary of the last n entries."""
        for i, e in enumerate(self.entries[-n:]):
            idx     = len(self.entries) - n + i
            pnl     = e.get("pnl")
            pnl_str = f"${pnl:+.2f}" if pnl is not None else "pending"
            label   = e.get("label") or "unlabelled"
            decs    = json.loads(e["completion"])
            actions = [f"{d['symbol']}:{d['action']}"
                       for d in decs if d.get("action") != "hold"][:5]
            print(f"[{idx:3d}] {label:>9} | P&L: {pnl_str:>10} | "
                  f"trades: {', '.join(actions) or 'all hold'}")
    def update_outcome(self, idx: int, equity_after: float):
        """
        Update the post-trade outcome for a logged episode.

        Args:
            idx          : index returned by log()
            equity_after : portfolio equity after trades

        Computes:
            pnl = equity_after - equity_before

        Notes:
        - If idx is None or invalid → silently ignore
        - Only updates once (prevents overwriting)
        """

        # --- Safety check ---
        if idx is None:
            return

        if not (0 <= idx < len(self.entries)):
            return

        entry = self.entries[idx]

        # --- Prevent double update ---
        if entry.get("equity_after") is not None:
            return

        entry["equity_after"] = equity_after
        entry["pnl"] = equity_after - entry["equity"]

        self._save_all()
    # ── Internal ───────────────────────────────────────────────────────────

    def _save_all(self):
        with open(self.save_path, "w", encoding="utf-8") as f:
            for e in self.entries:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")


print("Training data utilities loaded.")


### 1.14 Backtesting Infrastructure

> **Run this cell now, in order.** It only defines tools here; Task 4 uses them (4a collects training data, 4c compares the model before and after training).

Simulates N trading days using **historical prices** (yfinance) and **historical news**
(Alpaca, Drive-cached), with the agent loop making decisions each day.

#### Data flow per backtest day

```
fetch_historical_prices()           → daily closing prices (yfinance, Drive-cached)

For each trading day:
  1. llm_agent_decide(tok, model, state)
       Agent sees full universe + tool list, then:
       - Calls get_news(["NVDA","AAPL",...])  ← agent picks which stocks
         First call fetches from Alpaca & caches to Drive.
         Same-day re-runs hit the cache instantly (no API quota used).
       - Optionally calls get_rsi, get_macd, get_price_history
       - Outputs final JSON decision array

  2. SimulatedPortfolio.execute()   → simulate fills at that day's closing price

  3. TradingDataCollector.log()     → record conversation + decisions for SFT

  4. TradingDataCollector.update_outcome() → record next day's equity as outcome
```

#### Key classes

| Class | Purpose |
|-------|---------|
| `SimulatedPortfolio` | Tracks cash, positions, equity without any real orders |
| `fetch_historical_prices()` | Downloads/caches yfinance daily closes |
| `run_backtest()` | Orchestrates the full backtest loop |

#### After the backtest

```python
collector.auto_label()   # label good/bad based on PnL
collector.summary()      # print statistics
collector.review()       # inspect recent episodes
```
Then proceed to Task 4 to train the SFT model on the "good" episodes.


In [ ]:
# =============================================================================
# Backtesting Infrastructure
# =============================================================================

class SimulatedPortfolio:
    """In-memory paper portfolio for backtesting — no real orders placed."""

    def __init__(self, initial_cash: float = 100_000.0):
        self.initial_cash = initial_cash
        self.cash         = initial_cash
        self.positions    = {}   # symbol -> {"qty": float, "avg_price": float}

    def equity(self, prices: dict) -> float:
        return self.cash + sum(
            p["qty"] * prices.get(sym, p["avg_price"])
            for sym, p in self.positions.items()
        )

    def get_state(self, prices: dict, timestamp: int = None) -> dict:
        pos = []
        for sym, p in self.positions.items():
            price = prices.get(sym, p["avg_price"])
            pos.append({
                "symbol":          sym,
                "qty":             p["qty"],
                "market_value":    round(p["qty"] * price, 2),
                "avg_entry_price": round(p["avg_price"], 2),
            })
        return {
            "cash":              round(self.cash, 2),
            "equity":            round(self.equity(prices), 2),
            "buying_power":      round(self.cash * 2, 2),
            "positions":         pos,
            "allowed_symbols":   WHITELIST,
            "max_qty_per_trade": MAX_QTY_PER_TRADE,
            "timestamp_unix":    timestamp or int(time.time()),
        }

    def execute(self, decision: dict, prices: dict):
        action = decision.get("action", "hold")
        symbol = decision.get("symbol")
        qty    = decision.get("qty")

        if action not in ("buy", "sell") or not symbol or not qty:
            return None
        if symbol not in prices:
            return None

        price = prices[symbol]

        if action == "buy":
            cost = qty * price
            if cost > self.cash:
                return None
            self.cash -= cost
            if symbol in self.positions:
                old       = self.positions[symbol]
                total_qty = old["qty"] + qty
                avg       = (old["qty"] * old["avg_price"] + cost) / total_qty
                self.positions[symbol] = {"qty": total_qty, "avg_price": avg}
            else:
                self.positions[symbol] = {"qty": float(qty), "avg_price": price}
            return {"status": "filled", "symbol": symbol, "side": "buy",
                    "qty": qty, "price": round(price, 2)}

        elif action == "sell":
            pos = self.positions.get(symbol)
            if pos is None or pos["qty"] < qty:
                return None
            self.cash += qty * price
            pos["qty"] -= qty
            if pos["qty"] <= 0:
                del self.positions[symbol]
            return {"status": "filled", "symbol": symbol, "side": "sell",
                    "qty": qty, "price": round(price, 2)}

        return None


def fetch_historical_prices(symbols: list = None, period: str = "3mo") -> dict:
    """
    Download historical daily closing prices via yfinance (Drive-cached daily).
    Returns {"dates": [...], "prices": {sym: {date: price}}, "timestamp": ...}
    """
    if symbols is None:
        symbols = WHITELIST

    cache_path = os.path.join(CACHE_DIR, "historical_prices.json")
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cached = json.load(f)
        if time.time() - cached.get("timestamp", 0) < 86400:
            print(f"Loaded cached price data ({len(cached['dates'])} trading days)")
            return cached

    print(f"Downloading {period} of daily prices for {len(symbols)} symbols...")
    df = yf.download(symbols, period=period, progress=False)

    close_df = df["Close"] if isinstance(df.columns, pd.MultiIndex) else df[["Close"]]
    if not isinstance(df.columns, pd.MultiIndex):
        close_df.columns = [symbols[0]]

    dates  = [d.strftime("%Y-%m-%d") for d in close_df.index]
    prices = {}
    for sym in symbols:
        if sym in close_df.columns:
            sym_prices = {}
            for d, v in zip(dates, close_df[sym]):
                try:
                    fv = float(v)
                    if fv == fv:   # NaN check
                        sym_prices[d] = round(fv, 2)
                except (ValueError, TypeError):
                    pass
            prices[sym] = sym_prices

    result = {"dates": dates, "prices": prices, "timestamp": time.time()}
    with open(cache_path, "w") as f:
        json.dump(result, f)
    print(f"Cached {len(dates)} trading days to Drive")
    return result


def run_backtest(tok, model, collector=None, n_days: int = 20,
                 initial_cash: float = 100_000.0, verbose: bool = True) -> dict:
    """
    Replay N historical trading days using the agent decision loop.

    For each day:
      1. llm_agent_decide()  — agent calls tools (incl. get_news) on demand
      2. Execute simulated trades at that day's closing prices
      3. Log episode to collector (if provided)

    Args:
        tok          : tokenizer
        model        : base model (or LoRA fine-tuned model for comparison)
        collector    : TradingDataCollector instance (or None to skip logging)
        n_days       : number of most-recent trading days to replay
        initial_cash : starting portfolio cash
        verbose      : print per-day progress

    Returns:
        dict with equity_history, trades, total_return, final_equity
    """
    price_data = fetch_historical_prices()
    all_dates  = price_data["dates"]
    n_days     = min(n_days, len(all_dates))
    dates      = all_dates[-n_days:]

    portfolio      = SimulatedPortfolio(initial_cash)
    equity_history = []
    all_trades     = []
    entry_indices  = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"BACKTEST  {dates[0]}  →  {dates[-1]}  ({len(dates)} days)")
        print(f"Initial cash: ${initial_cash:,.0f}  |  Universe: {len(WHITELIST)} stocks")
        print(f"Agent: up to 6 tool-call turns per day")
        print(f"{'='*60}")

    for date in dates:
        day_prices = {
            sym: price_data["prices"].get(sym, {}).get(date)
            for sym in WHITELIST
        }
        day_prices = {k: v for k, v in day_prices.items() if v is not None}
        if len(day_prices) < 5:
            continue

        print(f"{date}:")
        # Agent decides; get_news tool calls are cached by date
        try:
            ts = int(datetime.datetime.strptime(date, "%Y-%m-%d").timestamp())
        except Exception:
            ts = int(time.time())

        state         = portfolio.get_state(day_prices, timestamp=ts)
        equity_before = state["equity"]

        decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
            tok, model, state, universe=WHITELIST, date=date
        )

        # Simulate trades
        day_trades = 0
        for d in decisions:
            result = portfolio.execute(d, day_prices)
            if result:
                all_trades.append({**result, "date": date})
                day_trades += 1

        equity_after = portfolio.equity(day_prices)
        equity_history.append({"date": date, "equity": round(equity_after, 2)})

        # Log episode
        if collector is not None:
            idx = collector.log(sft_prompt, sft_completion, decisions, equity_before, tool_calls_made)
            entry_indices.append(idx)
            if len(entry_indices) > 1:
                collector.update_outcome(entry_indices[-2], equity_after)

        pnl = equity_after - initial_cash
        if verbose:
            print(f"  Equity: ${equity_after:>10,.2f}  |  "
                  f"P&L: ${pnl:>+9,.2f}  |  Trades: {day_trades}")

    # Update last entry outcome
    if collector and entry_indices:
        final_eq = equity_history[-1]["equity"] if equity_history else initial_cash
        collector.update_outcome(entry_indices[-1], final_eq)

    final_equity = equity_history[-1]["equity"] if equity_history else initial_cash
    total_return = (final_equity - initial_cash) / initial_cash * 100

    if verbose:
        print(f"\n{'='*60}")
        print(f"RESULTS")
        print(f"  Period:  {dates[0]}  →  {dates[-1]}")
        print(f"  Initial: ${initial_cash:>12,.2f}")
        print(f"  Final:   ${final_equity:>12,.2f}")
        print(f"  Return:  {total_return:>+11.2f}%")
        print(f"  Trades:  {len(all_trades):>12d}")
        print(f"{'='*60}")

    return {
        "equity_history": equity_history,
        "trades":         all_trades,
        "total_return":   total_return,
        "final_equity":   final_equity,
    }


print("Backtesting infrastructure loaded.")


### 1.15 Test Agent Decision Engine

Runs a quick smoke test of `llm_agent_decide()` using only the tools that are
currently implemented (i.e. registered with a non-None `func`).

**At this point (before Task 2):**
- `get_news` is available — the agent can fetch financial news
- `get_price_history`, `get_rsi`, `get_macd`, `get_global_quote` are not yet available

After completing Task 2 and registering your tools, re-run this cell to see
the agent use all of your implementations.


In [ ]:
# Quick smoke test — only tools implemented so far will be available
print("Available tools before Task 2:")
list_tools()

print("\nRunning agent on 5-symbol subset...")
state = get_state(tc)
decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
    tok, model, state, universe=WHITELIST[:5]
)

print(f"\nDecisions ({len(decisions)}):")
for d in decisions:
    sym    = d.get("symbol", "?")
    act    = d["action"].upper()
    qty    = d.get("qty", "-")
    reason = d.get("reason", "")[:60]
    print(f"  [{sym}] {act:5s}  qty={str(qty):>4}  {reason}")

print(f"\nSFT prompt  : {len(sft_prompt):,} chars")
print(f"SFT completion: {len(sft_completion):,} chars")


### Task 1 Complete!

You now have all core infrastructure running:
- LLM model loaded
- Alpaca trading client connected
- News cached to Google Drive
- Decision engine ready
- Training data collector ready (for Task 4)
- Backtesting engine ready (for Task 4)

**Next: Implement Tasks 2-5 below.**

---

## Task 2: [TODO] Implement MCP Tool Functions

### Objective

Implement the **data retrieval tools** that the trading agent uses to research stocks
before making decisions. When the agent outputs a tool call, `dispatch_tool()` routes
it to your function — so the quality of your implementations directly determines
how well-informed the agent's decisions are.

### Tool Registry Reference

All tool functions you implement in this task must be registered using the `register_tool()`
function from **Section 1.10**. After you write each function, call:

```python
register_tool(
    name        = "your_tool_name",
    description = "brief description for the agent",
    params      = {"param": "type — description"},
    func        = your_function,
)
```

See Section 1.10 for the complete Tool Registry API documentation.

### How a tool call reaches your code

```
Agent outputs:
  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA", "time_period": 14}}</tool_call>

dispatch_tool("get_rsi", {"symbol": "NVDA", "time_period": 14})
  └─ _tool_registry["get_rsi"]["func"]("NVDA", time_period=14)
       └─ YOUR get_rsi("NVDA", time_period=14)   ← implement this
            └─ returns: "RSI(14) for NVDA: 62.3 (neutral) [2024-01-15]"
```

### [TODO] Required: implement and register these tools

**Implementation order:** Start with `get_news_for_symbols` (Section 1.9) as it demonstrates the complete pattern: API call → error handling → caching → formatting. Then implement the remaining functions following the same pattern.

| Function | Data source | What to implement |
|----------|------------|-------------------|
| `get_news_for_symbols(symbols, limit, date)` | **Alpaca** | News with caching |
| `get_price_history(symbol, period)` | **yfinance** | Provided: recent OHLCV table |
| `get_rsi(symbol, time_period)` | **Alpha Vantage** | RSI value + signal |
| `get_macd(symbol)` | **yfinance** | MACD, signal, histogram (computed from daily closes) |
| `get_global_quote(symbol)` | **Alpha Vantage** | Live price, change, volume |

> Alpha Vantage made `NEWS_SENTIMENT` and `MACD` premium-only, so news comes from Alpaca (Section 1.9)
> and MACD is computed from yfinance prices. RSI and GLOBAL_QUOTE still work with the free Alpha Vantage key.

After implementing each function you **must** call `register_tool()` to update the
registry — otherwise the agent won't know the tool is ready.

### Alpha Vantage quick reference

```python
# RSI
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "RSI", "symbol": symbol,
    "interval": "daily", "time_period": time_period,
    "series_type": "close", "apikey": ALPHA_VANTAGE_API_KEY,
})
data = resp.json()
# data["Technical Analysis: RSI"] -> {date: {"RSI": "65.32"}, ...}

# Global Quote
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "GLOBAL_QUOTE", "symbol": symbol,
    "apikey": ALPHA_VANTAGE_API_KEY,
})
# data["Global Quote"] -> {"05. price": "182.63", "09. change": "+1.20", ...}
```

### yfinance quick reference

```python
import yfinance as yf
ticker = yf.Ticker("AAPL")
hist   = ticker.history(period="5d")   # pandas DataFrame
# columns: Open, High, Low, Close, Volume
# index:   DatetimeIndex

# MACD from daily closes (Alpha Vantage's MACD is premium-only)
close  = ticker.history(period="6mo")["Close"]
macd   = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
signal = macd.ewm(span=9, adjust=False).mean()
hist   = macd - signal
```

### Optional: add your own custom tool

Implement any additional data source (earnings, options flow, sector ETF performance,
macroeconomic indicators, etc.) and register it so the agent can call it automatically:

```python
def get_earnings(symbol: str) -> str:
    """Fetch latest quarterly earnings from Alpha Vantage."""
    # function="EARNINGS" -> data["quarterlyEarnings"][0]
    ...

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# Agent now calls get_earnings automatically when it wants earnings data
```

### Grading criteria

- [ ] `get_news_for_symbols` implements full caching + API call pattern. *Evidence: Test 1.9 (the repeated query prints `Cache hit`).*
- [ ] At least **2** Alpha Vantage functions implemented; each handles API errors gracefully (try/except); all registered with `register_tool(..., func=your_fn)`. *Evidence: Test Task 2 output and `list_tools()` in 1.15.*
- `get_price_history` is provided. Optional custom tools are welcome; describe them in your report.


In [ ]:
# =============================================================================
# TODO: Task 2 — Implement MCP Tool Functions
# Each function is called by dispatch_tool() when the agent requests it.
# After implementing a function, update its registry entry so the agent can use it.
# =============================================================================


# ── Required: get_price_history ───────────────────────────────────────────────

def get_price_history(symbol: str, period: str = "5d") -> str:
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period=period)

        if hist.empty:
            return f"No price history available for {symbol}."

        lines = [f"Price history for {symbol} ({period}):"]

        for date, row in hist.iterrows():
            d = date.strftime("%Y-%m-%d")
            o = row["Open"]
            h = row["High"]
            l = row["Low"]
            c = row["Close"]
            v = int(row["Volume"])

            lines.append(
                f"  {d}: O={o:.2f} H={h:.2f} L={l:.2f} C={c:.2f} V={v}"
            )

        return "\n".join(lines)

    except Exception as e:
        return f"Price history fetch failed for {symbol}: {e}"


register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = get_price_history,   # update func here after implementing above
)


# ── Required: get_rsi ─────────────────────────────────────────────────────────

def get_rsi(symbol: str, interval: str = "daily", time_period: int = 14) -> str:
    """
    Fetch RSI (Relative Strength Index) for a symbol from Alpha Vantage.

    RSI > 70  →  overbought (potential sell signal)
    RSI < 30  →  oversold   (potential buy signal)
    30–70     →  neutral

    Alpha Vantage params:
        function="RSI", symbol, interval="daily",
        time_period=14, series_type="close", apikey

    Returns:
        e.g. "RSI(14) for NVDA: 68.5 (neutral — approaching overbought) [2024-01-15]"
    """
    # TODO: implement

    raise NotImplementedError("TODO: Implement get_rsi()")


register_tool(
    name        = "get_rsi",
    description = "Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params      = {"symbol": "str", "time_period": "int (default 14)"},
    func        = get_rsi,
)


# ── Required: get_macd ────────────────────────────────────────────────────────

def get_macd(symbol: str, interval: str = "daily") -> str:
    """
    Fetch MACD (Moving Average Convergence Divergence) for a symbol.

    MACD > Signal  →  bullish crossover
    MACD < Signal  →  bearish crossover

    Alpha Vantage's MACD is premium-only, so compute it from yfinance daily closes
    (see "yfinance quick reference" in the Task 2 description):
        MACD   = EMA12(close) - EMA26(close)
        Signal = EMA9(MACD)
        Hist   = MACD - Signal

    Returns:
        e.g. "MACD for NVDA [2024-01-15]: MACD=0.1234, Signal=0.0987, Hist=0.0247 (bullish)"
    """
    # TODO: implement
    raise NotImplementedError("TODO: Implement get_macd()")


register_tool(
    name        = "get_macd",
    description = "Get MACD trend indicator via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = get_macd,
)


# ── Required: get_global_quote ────────────────────────────────────────────────

def get_global_quote(symbol: str) -> str:
    """
    Fetch real-time price quote for a symbol from Alpha Vantage.

    Alpha Vantage params:
        function="GLOBAL_QUOTE", symbol, apikey

    Response key: data["Global Quote"]
        "05. price", "09. change", "10. change percent", "06. volume"

    Returns:
        e.g. "Quote AAPL: $186.10 | change: +1.20 (+0.65%) | vol: 52,341,200"
    """
    # TODO: implement
    raise NotImplementedError("TODO: Implement get_global_quote()")


register_tool(
    name        = "get_global_quote",
    description = "Get real-time price quote via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = get_global_quote,
)


# =============================================================================
# =============================================================================
# Optional C: search_market_news -- agent-driven stock discovery
# =============================================================================
# Instead of researching pre-defined tickers, this tool fetches the latest
# market-wide news from Alpaca WITHOUT specifying any symbols. The agent reads the
# headlines and discovers interesting companies on its own.
#
# Alpaca News API (no symbols= argument; same URL and headers as Section 1.9):
#   params: limit=20 (max 50); no topic filter and no sentiment scores
#   response: data["news"] -> list of articles, each with "symbols":
#     ["NVDA", "AMD"]
#
# def search_market_news(topic: str = "technology") -> str:
#     """
#     Fetch the latest market-wide news; return top mentioned tickers + headlines.
#
#     Args:
#         topic : optional keyword (e.g. "technology", "earnings", "ipo");
#                 Alpaca has no topic filter, so filter the headlines yourself
#     Returns:
#         Formatted string, e.g.:
#         "Top mentioned tickers for topic=technology:\n"
#         "  NVDA: 5 articles\n"
#         "  MSFT: 3 articles\n"
#         "Headlines:\n  - Nvidia announces next-gen GPU ...\n"
#     """
#     url = "https://data.alpaca.markets/v1beta1/news"
#     params = {"limit": 20}   # headers: your Alpaca keys, as in Section 1.9
#     # TODO: call API, filter by topic, count symbols across articles
#     raise NotImplementedError("TODO: implement search_market_news()")
#
# register_tool(
#     name        = "search_market_news",
#     description = (
#         "Fetch the latest market-wide news (no ticker needed) and the most "
#         "mentioned tickers. Optional topic keyword, e.g. technology, earnings, ipo."
#     ),
#     params      = {"topic": "str"},
#     func        = search_market_news,
# )

# Optional: add your own custom tool
# =============================================================================
#
# def get_MY_tool(symbol: str) -> str:
#     """Your description here."""
#     # ... implementation ...
#     return formatted_string
#
# register_tool(
#     name        = "get_MY_tool",
#     description = "One-line description for the agent",
#     params      = {"symbol": "str — stock ticker"},
#     func        = get_MY_tool,
# )


# =============================================================================
# get_extra_context
# =============================================================================
# This function aggregates data from your MCP tools into a single context string
# that gets passed to llm_agent_decide() in the trading loop (Task 5, Step 3).
#
# It calls get_price_history, get_rsi, get_macd, and get_global_quote for each
# symbol, collects the results, and joins them into one big string.
# If you implemented Optional C (search_market_news), call it here too.
# =============================================================================

def get_extra_context(symbols: list) -> str:
    """
    Aggregate outputs from all implemented MCP tools for the given symbols.

    For each symbol in the list, call:
      - get_price_history(symbol)
      - get_rsi(symbol)
      - get_macd(symbol)
      - get_global_quote(symbol)

    Collect results into a single formatted string. Each tool call should be
    wrapped in try/except so one failure doesn't block the rest.

    If you implemented search_market_news (Optional C), also call it with a
    relevant topic (e.g. "technology") and append the result.

    Args:
        symbols : list of ticker strings, e.g. ["AAPL", "NVDA"]

    Returns:
        Combined string of all tool outputs, separated by newlines.
        Returns "" if all calls fail.

    Example output:
        === Extra Context for AAPL ===
        Price history for AAPL (5d):
          2024-01-15: O=185.00 H=186.50 L=184.20 C=186.10 V=52341200
          ...
        RSI(14) for AAPL: 68.5 (neutral)
        MACD for AAPL: MACD=0.12, Signal=0.10, Hist=0.02 (bullish)
        Quote AAPL: 86.10 | change: +1.20 (+0.65%) | vol: 52,341,200
        ============================================

    Hints:
        tools = [get_price_history, get_rsi, get_macd, get_global_quote]
        for symbol in symbols:
            for tool in tools:
                try: results.append(tool(symbol))
                except: pass
    """
    # TODO: implement
    raise NotImplementedError("TODO: implement get_extra_context()")


### Test Task 2

In [ ]:
# Test your implementations
print("Testing MCP Tool Functions...")
print("=" * 60)

for func_name, func in [("get_rsi", get_rsi), ("get_macd", get_macd),
                          ("get_price_history", get_price_history),
                          ("get_global_quote", get_global_quote)]:
    try:
        result = func("AAPL")
        print(f"{func_name}('AAPL'):")
        print(f"  {result[:200]}")
    except NotImplementedError:
        print(f"{func_name}: Not yet implemented")
    except Exception as e:
        print(f"{func_name}: Error - {e}")

print("" + "=" * 60)

# Test get_extra_context (aggregates all tool outputs)
try:
    extra = get_extra_context(["AAPL"])
    assert isinstance(extra, str), f"Expected str, got {type(extra)}"
    print(f"get_extra_context(['AAPL']):")
    print(f"  Length: {len(extra)} chars")
    print(f"  Preview: {extra[:300]}")
except NotImplementedError:
    print("get_extra_context: Not yet implemented")
except Exception as e:
    print(f"get_extra_context: Error - {e}")

# Test that get_extra_context handles multiple symbols
try:
    extra_multi = get_extra_context(["AAPL", "NVDA"])
    assert isinstance(extra_multi, str), f"Expected str, got {type(extra_multi)}"
    assert len(extra_multi) >= len(extra), "Multi-symbol result should be >= single-symbol"
    print(f"get_extra_context(['AAPL', 'NVDA']):")
    print(f"  Length: {len(extra_multi)} chars")
except NotImplementedError:
    pass
except Exception as e:
    print(f"get_extra_context multi-symbol: Error - {e}")

print("" + "=" * 60)
print("Task 2 testing complete.")


---
## Task 3: [TODO] RAG Memory System

### Objective
Implement a memory system that allows the trading agent to **learn from past trades**.

Two components:
1. **`TradingMemory`** class - stores situation-reflection pairs, retrieves similar experiences using BM25
2. **`reflect_on_trade()`** function - uses LLM to generate a lesson learned after a trade

### How it works

```
Trade executed -> Wait N loops -> Compare equity -> LLM reflection -> Store in memory
                                                                           |
Next decision <- Retrieve similar past reflections via BM25 <--------------+
```

### BM25 Retrieval
[BM25](https://en.wikipedia.org/wiki/Okapi_BM25) is a keyword-based ranking function:
- Scores documents based on **term frequency** and **inverse document frequency**
- The `rank_bm25` library provides `BM25Okapi`

```python
from rank_bm25 import BM25Okapi

corpus = ["apple stock rises today", "tesla earnings beat expectations"]
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)
scores = bm25.get_scores("apple earnings".lower().split())
# scores = [0.45, 0.32]  -- first doc scores higher for "apple"
```

### Grading Criteria
- [ ] TradingMemory correctly stores and loads memories; BM25 retrieval returns relevant past reflections; edge cases handled (empty memory, no matching results). *Evidence: Test Task 3, and the first Task 5 iterations (memory is still empty there).*
- [ ] reflect_on_trade generates meaningful short reflections: each names the stock and the action and says whether the trade gained or lost. *Evidence: Test Task 3.*


### 3a. TradingMemory Class

In [ ]:
class TradingMemory:
    """
    BM25-based memory system for storing and retrieving trading reflections.
    Each memory entry: {"situation": str, "reflection": str, "timestamp": int}
    """

    def __init__(self, memory_file: str = "trading_memory.json"):
        """
        Initialize the memory system.

        TODO:
        - Store memory_file path as self.memory_file
        - Initialize self.memories as empty list
        - Load existing memories from file if it exists
          (use json.load, handle file-not-found and parse errors gracefully)
        """
        # TODO: Implement
        if memory_file is None:
            memory_file = os.path.join(CACHE_DIR, "trading_memory.json")

        self.memory_file = memory_file
        self.memories = []
#
        if os.path.exists(self.memory_file):
            try:
                with open(self.memory_file, "r", encoding="utf-8") as f:
                    self.memories = json.load(f)
                print(f"Loaded {len(self.memories)} trading memories")
            except Exception as e:
                print(f"Memory file corrupted, starting fresh: {e}")
                self.memories = []

    def _save(self):
        """Persist memories to disk as JSON."""
        with open(self.memory_file, "w", encoding="utf-8") as f:
            json.dump(self.memories, f, ensure_ascii=False, indent=2)

    def add_memory(self, situation: str, reflection: str):
        """
        Store a new situation-reflection pair.

        TODO:
        - Append {"situation": situation, "reflection": reflection,
                   "timestamp": int(time.time())} to self.memories
        - Call self._save()
        """
        # TODO: Implement
        self.memories.append({
            "situation": situation,
            "reflection": reflection,
            "timestamp": int(time.time())
        })

        self._save()

    def get_memories(self, query: str, n_matches: int = 2) -> list:
        """
        Retrieve the most similar past reflections using BM25.

        Args:
            query: Current situation text (search query)
            n_matches: Number of results to return

        Returns:
            List of reflection strings from the most similar past situations.
            Only include results where BM25 score > 0.

        TODO:
        1. If self.memories is empty, return []
        2. Build corpus: [m["situation"] for m in self.memories]
        3. Tokenize: [doc.lower().split() for doc in corpus]
        4. Tokenize query: query.lower().split()
        5. Create BM25Okapi(tokenized_corpus)
        6. Get scores: bm25.get_scores(tokenized_query)
        7. Sort indices by score (descending), take top n_matches
        8. Return reflections for those indices (only where score > 0)
        """
        # TODO: Implement
        raise NotImplementedError("TODO: implement get_memories()")

### 3b. Trade Reflection Function

In [ ]:
def reflect_on_trade(tok, model, situation: str, decision: dict,
                     prev_equity: float, curr_equity: float) -> str:
    """
    Use LLM to generate a short reflection on a trade outcome.

    Args:
        tok: Tokenizer
        model: LLM model
        situation: Market situation when decision was made
        decision: The trading decision dict
        prev_equity: Portfolio equity before the trade
        curr_equity: Portfolio equity after the trade

    Returns:
        A short reflection string (max 300 chars).
        Return "No reflection generated." if generation fails.

    TODO:
    1. Calculate P&L: pnl = curr_equity - prev_equity
       Calculate P&L pct: pnl_pct = (pnl / prev_equity) * 100
    2. Build a prompt asking the LLM to write a 1-2 sentence lesson learned:
       - System: "You are a trading reflection assistant..."
       - User: Include the decision (JSON), profit/loss amount, situation summary
       - Use prompt format: <|system|>...\n<|user|>...\n<|assistant|>\n
    3. Generate with max_new_tokens=100, do_sample=False
    4. Extract the assistant response, truncate to 300 chars
    """
    # TODO: Implement

    raise NotImplementedError("TODO: implement reflect_on_trade()")

### Test Task 3

In [ ]:
# Test TradingMemory
try:
    test_mem = TradingMemory("test_memory.json")

    test_mem.add_memory(
        situation="AAPL bullish news, strong earnings, bought 10 shares",
        reflection="Buying on strong earnings news was profitable."
    )
    test_mem.add_memory(
        situation="TSLA bearish sentiment, missed earnings, sold 5 shares",
        reflection="Selling before bearish news helps protect capital."
    )
    test_mem.add_memory(
        situation="MSFT neutral news, market sideways, held position",
        reflection="Holding during uncertain times avoided transaction costs."
    )

    results = test_mem.get_memories("AAPL earnings report bullish", n_matches=2)
    print("Memory retrieval test:")
    print(f"  Query: 'AAPL earnings report bullish'")
    print(f"  Retrieved {len(results)} memories:")
    for i, r in enumerate(results, 1):
        print(f"    {i}. {r}")

    if os.path.exists("test_memory.json"):
        os.remove("test_memory.json")
    print("\nTradingMemory works!")
except NotImplementedError:
    print("Task 3a not yet implemented")
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()

# Test reflect_on_trade
try:
    reflection = reflect_on_trade(
        tok, model,
        situation="AAPL bullish news, bought 5 shares at $280",
        decision={"action": "buy", "symbol": "AAPL", "qty": 5},
        prev_equity=100000.0, curr_equity=100150.0
    )
    print(f"\nReflection test: {reflection}")
    print("reflect_on_trade works!")
except NotImplementedError:
    print("\nTask 3b (reflect_on_trade) not yet implemented")
except Exception as e:
    print(f"\nError: {e}")

---
## Task 4: [TODO] SFT Fine-tuning with LoRA

### Objective
Fine-tune the Qwen 1.5B model on **backtested trading data** using LoRA (Low-Rank Adaptation), then compare performance.

### Flow

```
Step 1: Backtest with BASE model
         Simulate ~60 trading days on historical prices
         LLM makes decisions each day (no real money)
         All (prompt, decisions, equity_change) logged automatically
                         │
Step 2: Auto-label data
         equity went UP   → label "good"
         equity went DOWN → label "bad"
                         │
Step 3: SFT fine-tune (YOUR IMPLEMENTATION)
         Filter "good" examples
         Tokenize as (prompt, completion) pairs
         LoRA: only train <0.1% of parameters
         Save adapter to Drive
                         │
Step 4: Backtest with FINE-TUNED model
         Same historical data, same starting cash
         Compare returns: base vs fine-tuned
```

### What you need to do

1. **`prepare_sft_dataset()`** - Filter labeled data, tokenize into training format
2. **`get_lora_model()`** - Configure LoRA adapters on the base model
3. **`train_sft()`** - Run the training loop
4. Compare backtest results

### Key Concepts

**LoRA** adds small trainable matrices to attention layers while freezing the base model:
- `r=8`: rank of the low-rank matrices (higher = more capacity but slower)
- `lora_alpha=16`: scaling factor (typically 2x r)
- `target_modules=["q_proj", "v_proj"]`: which layers to adapt
- Only ~0.07% of parameters are trainable → fast training on Colab

**SFT (Supervised Fine-Tuning)**: Train the model to reproduce "good" trading decisions.
The loss is only computed on the **completion** (decisions), not the prompt.

### Grading Criteria
- [ ] `prepare_sft_dataset` correctly filters and tokenizes data; LoRA config is reasonable (r, alpha, target modules); the training loop runs without errors and every epoch's loss is a normal number. *Evidence: 4c output: the dataset size matches the GOOD count (explain any difference in your report), the trainable-parameter line, the loss per epoch.*
- [ ] 4a and 4c are provided cells: run them, then write your observations and explanations in the training-strategy part of your report. The note at the top of 4a and of 4c says what to cover.

### 4a. Run Backtest & Collect Training Data

Run a backtest with the **base model** on historical data. This collects trading decisions and outcomes
without spending any paper money. Data is auto-labeled based on P&L.

> **Provided cell: run it, then explain what you see in your report.** It replays your agent with the base model over the last ~60 trading days (`run_backtest`, 1.14); the collector (1.13) logs the valid days, and `auto_label` marks each one GOOD or BAD from the next day's profit or loss. In your report: how many days were logged, the GOOD/BAD split, whether 1–2 sample labels make sense (the inspect cell below helps), and why you changed the labeling rule if you did.

In [ ]:
# =============================================================================
# Run backtest with BASE model to collect training data
# =============================================================================

# Initialize a fresh collector (clears previous data)
import os as _os
_sft_path = _os.path.join(CACHE_DIR, "sft_data.jsonl")
if _os.path.exists(_sft_path):
    _os.remove(_sft_path)
    print("Cleared previous training data")

collector = TradingDataCollector()

# Run backtest - this takes a few minutes
print("Running backtest with BASE model...")
print("(Each day = 1 LLM inference, ~15-30 sec on T4)\n")
base_results = run_backtest(tok, model, collector=collector, n_days=60)  # 3 months of prices ≈ 63 trading days

# Auto-label: good if equity increased, bad if decreased
collector.auto_label(threshold=0.0)

print("\n--- Training Data Summary ---")
collector.summary()

# Review labeled entries
print("\nRecent entries:")
collector.review(n=10)

# Manual override example (uncomment to use):
# collector.set_label(0, "good")   # override entry 0
# collector.set_label(3, "bad")    # override entry 3

print(f"\nGood examples for training: {len(collector.get_labeled('good'))}")
print(f"Bad examples (not used in SFT): {len(collector.get_labeled('bad'))}")

### Optional: Inspect & Edit SFT Training Data

After the backtest, `sft_data.jsonl` contains all logged (prompt, completion, label) pairs.
Use the cells below to review entries, remove low-quality ones, or upload a hand-edited version
before running `train_sft()`.


In [ ]:
# =============================================================================
# OPTIONAL: Inspect & Edit sft_data.jsonl
# Run each sub-block independently as needed.
# =============================================================================

import os, json
from google.colab import files

sft_path = os.path.join(CACHE_DIR, "sft_data.jsonl")

# -- 1. View all SFT entries -----------------------------------------------
if os.path.exists(sft_path):
    with open(sft_path) as f:
        entries = [json.loads(l) for l in f if l.strip()]
    print(f"Found {len(entries)} SFT entries:")
    good = [e for e in entries if e.get('label') == 'good']
    bad  = [e for e in entries if e.get('label') == 'bad']
    print(f"  good: {len(good)}  |  bad: {len(bad)}\n")
    for i, e in enumerate(entries):
        label  = e.get('label', '?')
        pnl    = e.get('pnl', '?')
        compl  = e.get('completion', '')[:100].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      completion: {compl}")
        print()
else:
    print("No sft_data.jsonl found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(sft_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment after editing locally:
# uploaded = files.upload()   # select your edited sft_data.jsonl
# for fname, content in uploaded.items():
#     with open(sft_path, 'wb') as f:
#         f.write(content)
# with open(sft_path) as f:
#     entries = [json.loads(l) for l in f if l.strip()]
# print(f"Reloaded {len(entries)} SFT entries from uploaded file.")

# -- 4. Remove bad entries (keep only good for training) ------------------
# Uncomment to filter:
# if os.path.exists(sft_path):
#     with open(sft_path) as f:
#         entries = [json.loads(l) for l in f if l.strip()]
#     good_only = [e for e in entries if e.get('label') == 'good']
#     with open(sft_path, 'w') as f:
#         for e in good_only:
#             f.write(json.dumps(e) + '\n')
#     print(f"Kept {len(good_only)} good entries; removed {len(entries)-len(good_only)} bad.")


### 4b. Implement SFT Fine-tuning

In [ ]:
# =============================================================================
# TODO: [IMPLEMENTATION REQUIRED] - SFT Fine-tuning with LoRA
# =============================================================================


def prepare_sft_dataset(collector, tok, max_length=2048):
    """
    Convert labeled "good" trading decisions into a tokenized SFT dataset.

    Args:
        collector: TradingDataCollector with labeled entries
        tok: Tokenizer
        max_length: Max sequence length for tokenization

    Returns:
        datasets.Dataset with columns: ["input_ids", "attention_mask", "labels"]

    TODO:
    1. Get "good" entries: collector.get_labeled("good")
    2. If empty, raise ValueError("No 'good' labeled data found")
    3. For each entry:
       a. full_text = entry["prompt"] + entry["completion"]
       b. Tokenize: tok(full_text, truncation=True, max_length=max_length,
                        padding="max_length", return_tensors="pt")
       c. Create labels = input_ids.clone()
       d. Mask the prompt tokens: prompt_ids = tok(entry["prompt"])["input_ids"]
          labels[:len(prompt_ids)] = -100  (so loss is only on completion)
    4. Collect all into lists, return Dataset.from_dict({
           "input_ids": [...], "attention_mask": [...], "labels": [...]
       })
    """
    # TODO: Implement
    raise NotImplementedError("TODO: implement prepare_sft_dataset()")


def get_lora_model(base_model):
    """
    Wrap the base model with LoRA adapters for fine-tuning.

    Returns:
        PEFT model with LoRA adapters attached.

    TODO:
    1. Create LoraConfig:
       - r=8
       - lora_alpha=16
       - target_modules=["q_proj", "v_proj"]
       - lora_dropout=0.05
       - bias="none"
       - task_type=TaskType.CAUSAL_LM
    2. Apply: peft_model = get_peft_model(base_model, config)
    3. Print trainable params: peft_model.print_trainable_parameters()
    4. Return peft_model
    """
    # TODO: Implement
    raise NotImplementedError("TODO: implement get_lora_model()")


def train_sft(peft_model, dataset, tok, epochs=3, lr=2e-4, batch_size=1):
    """
    Train the LoRA model on the SFT dataset.

    Args:
        peft_model: Model with LoRA adapters
        dataset: Dataset from prepare_sft_dataset()
        tok: Tokenizer
        epochs: Number of training epochs
        lr: Learning rate
        batch_size: Batch size (1 recommended for Colab)

    TODO:
    1. Create optimizer: torch.optim.AdamW(peft_model.parameters(), lr=lr)
    2. Set peft_model.train()
    3. Training loop:
       for epoch in range(epochs):
           total_loss = 0
           for i in range(0, len(dataset), batch_size):
               batch = dataset[i:i+batch_size]
               input_ids = torch.tensor(batch["input_ids"]).to(peft_model.device)
               attention_mask = torch.tensor(batch["attention_mask"]).to(peft_model.device)
               labels = torch.tensor(batch["labels"]).to(peft_model.device)

               outputs = peft_model(input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    labels=labels)
               loss = outputs.loss
               loss.backward()
               optimizer.step()
               optimizer.zero_grad()
               total_loss += loss.item()

           avg_loss = total_loss / max(1, len(dataset) // batch_size)
           print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

    4. Save adapter:
       save_dir = os.path.join(CACHE_DIR, "lora_adapter")
       peft_model.save_pretrained(save_dir)
       print(f"LoRA adapter saved to {save_dir}")

    5. Set peft_model.eval()
    """
    # TODO: Implement
    raise NotImplementedError("TODO: implement train_sft()")

### 4c. Re-run Backtest & Compare

Run the same backtest with the **fine-tuned model** and compare returns against the base model.

> **Provided cell: run it, then explain what you see in your report.** It trains LoRA with your 4b functions, then backtests the base model and the fine-tuned model on the last 20 trading days and prints the return and number of trades for each. In your report: how the two models differ (return, trades, whether they make real decisions each day), what you think caused it, and which conclusions you would not trust, and why.

In [ ]:
# =============================================================================
# Compare base vs fine-tuned model via backtest
# =============================================================================
print("=" * 60)
print("SFT Fine-tuning Pipeline")
print("=" * 60)

collector = TradingDataCollector()
good_count = len(collector.get_labeled("good"))

if good_count == 0:
    print("\nNo 'good' labeled data found!")
    print("Run cell 4a first to backtest and collect training data.")
else:
    print(f"\nTraining on {good_count} 'good' examples...")

    try:
        # Step 1: Prepare dataset
        dataset = prepare_sft_dataset(collector, tok)
        print(f"Dataset ready: {len(dataset)} examples")

        # Step 2: Create LoRA model
        lora_model = get_lora_model(model)

        # Step 3: Train
        train_sft(lora_model, dataset, tok)
        print("\nTraining complete!")

        # Step 4: Backtest base model (comparison baseline)
        # Comment this part if you only want to run Fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Base Model (comparison baseline)")
        print("=" * 60)
        # get_peft_model() adds LoRA to `model` in place, so `model` is no longer
        # the base model here. disable_adapter() turns LoRA off for this baseline run.
        with lora_model.disable_adapter():
            base_results = run_backtest(tok, lora_model, collector=None, n_days=20)

        # Step 5: Backtest with fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Fine-tuned Model")
        print("=" * 60)
        ft_results = run_backtest(tok, lora_model, collector=None, n_days=20)

        # Step 6: Compare
        print("\n" + "=" * 60)
        print("COMPARISON: Base Model vs Fine-tuned Model")
        print("=" * 60)
        print(f"  Base model return:       {base_results['total_return']:>+8.2f}%")
        print(f"  Fine-tuned model return: {ft_results['total_return']:>+8.2f}%")
        diff = ft_results['total_return'] - base_results['total_return']
        print(f"  Improvement:             {diff:>+8.2f}%")
        print(f"")
        print(f"  Base trades:             {len(base_results['trades']):>8d}")
        print(f"  Fine-tuned trades:       {len(ft_results['trades']):>8d}")
        print("=" * 60)

        if diff > 0:
            print("\nFine-tuned model outperformed the base model!")
        elif diff < 0:
            print("\nBase model performed better. Consider:")
            print("  - Collecting more training data (increase n_days)")
            print("  - Adjusting labeling threshold")
            print("  - Tuning LoRA hyperparameters (r, alpha, lr)")
        else:
            print("\nBoth models performed equally.")

    except NotImplementedError as e:
        print(f"\nNot yet implemented: {e}")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc()

---
## Task 5: [TODO] Full Trading Loop Integration

### Objective
Wire Tasks 2-3 together with the pre-written infrastructure into a complete trading loop, using the **fine-tuned model** from Task 4.

Each iteration:
1. Get current account state
2. Fetch financial news (from Drive cache)
3. Fetch extra context from your MCP tool functions (Task 2)
4. Check pending trades for reflection (after delay)
5. Retrieve similar past experiences from memory
6. Make LLM-powered decisions
7. Execute orders
8. Queue trades for future reflection

### Important Notes
- The loop runs for `n_iterations` (not infinite)
- Use `time.sleep(LOOP_SECONDS)` between iterations
- `pending_reflections` tracks trades awaiting reflection
- A trade is ready for reflection when `loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS`

### Grading Criteria
- [ ] All steps are correctly wired together; error handling prevents one failed step from crashing the loop; `extra_context` from Task 2 is passed to `llm_agent_decide()`; memory from Task 3 is used for reflections and retrieval; orders are only submitted for buy/sell decisions (not hold). *Evidence: at least 10 consecutive iterations, each showing the memory retrieval and the Task 2 data, and at least one order with its status.*
- [ ] Live week (11/9–11/13): the output of every trading day, plus the Alpaca order history and final balance. *Evidence: the daily logs in `trading_logs/` on Drive; orders and the final balance are checked in your Alpaca account.*

In [ ]:
def run_trading_loop(tc, tok, model, memory=None, collector=None,
                     n_iterations=5):
    """
    Execute the main trading loop.

    Args:
        tc: Alpaca TradingClient
        tok: Tokenizer
        model: LLM model (use fine-tuned lora_model from Task 4)
        memory: TradingMemory instance (None to disable memory)
        collector: TradingDataCollector instance (None to disable data collection)
        n_iterations: Number of loop iterations to run
    """
    pending_reflections = []
    pending_outcomes = []  # track entries for collector outcome updates

    for loop_count in range(1, n_iterations + 1):
        print(f"\n{'='*60}")
        print(f"LOOP #{loop_count} / {n_iterations} | Pending reflections: {len(pending_reflections)}")
        print('='*60)

        # --------------------------------------------------------------
        # TODO Step 1: Get current account state
        # state = get_state(tc)
        # Print equity, cash, positions
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 2: Fetch financial news (cached to Drive)
        # news_context = get_news_for_symbols(symbols)   # a few tickers (see Step 3), not all 100
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 3: Fetch extra context from your MCP tools (Task 2)
        # try:
        #     extra_context = get_extra_context(symbols)
        #     # symbols = a few tickers (e.g. your positions + 3-5 from the news).
        #     # All 100 would need ~300 Alpha Vantage calls per loop (free tier: 25/day).
        # except:
        #     extra_context = ""
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 4: Check pending reflections
        # Find trades where loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS
        # For each ready trade:
        #   - Call reflect_on_trade(...)
        #   - Store in memory: memory.add_memory(trade["situation"], reflection)
        #   - Remove from pending_reflections
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 4.5: Update collector outcomes
        # For entries in pending_outcomes where
        #   loop_count - entry["loop"] >= REFLECTION_DELAY_LOOPS:
        #   if collector:
        #       collector.update_outcome(entry["idx"], state["equity"])
        #       remove from pending_outcomes
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 5: Retrieve past experiences from memory
        # If memory is not None:
        #   - Build a situation summary from news + state
        #   - past_reflections = memory.get_memories(summary, MEMORY_TOP_K)
        # Else: past_reflections = []
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 6: Make trading decisions using the MCP agent
        # decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
        #     tok, model, state, universe=WHITELIST
        # )
        # 4 return values. The model only sees its prompt: to use extra_context
        # and past_reflections, pass them into llm_agent_decide() (e.g. two new parameters).
        # Print each decision
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 6.5: Log to data collector
        # if collector is not None:
        #     idx = collector.log(sft_prompt, sft_completion, decisions,
        #                         state["equity"], tool_calls_made)
        #     pending_outcomes.append({"idx": idx, "loop": loop_count})
        # --------------------------------------------------------------


        # --------------------------------------------------------------
        # TODO Step 7: Execute orders
        # For each decision with action "buy" or "sell":
        #   try: result = submit_order(tc, d); print result
        #   except: print error
        #   if memory: queue for reflection in pending_reflections
        #     {"loop": loop_count, "situation": ...,
        #      "decision": d, "equity": state["equity"]}
        # For hold: just print the reason
        # --------------------------------------------------------------
        raise NotImplementedError("TODO: implement run_trading_loop()")

### Run the Trading Loop

Use the **fine-tuned model** (`lora_model`) from Task 4 for live paper trading.

#### Live week (11/9–11/13)

- Each trading day, run the loop during market hours (9:30–16:00 ET), ideally for 2–4 hours (longer if you can; set `LOOP_SECONDS` and `n_iterations` to match).
- **Do not rerun 4a or 4c on live days.** 4a clears your training data and 4c retrains the model, which would change your frozen model. Run Task 1 to Task 3 and the `run_trading_loop` cell, then the cell below: it loads the adapter Task 4 saved on Drive by itself.
- **The daily log is kept for you.** Everything the cell below prints is also saved to `CACHE_DIR/trading_logs/<date>.log` on Drive, so make your loop print each decision and order clearly. You submit these logs on 11/19, and your report reflects on the week.

In [ ]:
# Initialize memory system
try:
    memory = TradingMemory(os.path.join(CACHE_DIR, "trading_memory.json"))  # keep on Drive
    print(f"Memory loaded: {len(memory.memories)} past entries")
except:
    memory = None
    print("Memory not available (implement Task 3 first)")

# Initialize data collector
collector = TradingDataCollector()

# Use fine-tuned model from Task 4. On live days (4a and 4c skipped), load the adapter Task 4 saved on Drive.
try:
    trading_model = lora_model
    model_note = "fine-tuned LoRA model from Task 4 (this session)"
    print("Using fine-tuned LoRA model from Task 4")
except NameError:
    try:
        from peft import PeftModel
        trading_model = PeftModel.from_pretrained(model, os.path.join(CACHE_DIR, "lora_adapter"))
        trading_model.eval()
        model_note = "fine-tuned LoRA adapter loaded from Drive"
        print("Loaded the fine-tuned LoRA adapter saved on Drive")
    except Exception as e:
        trading_model = model
        model_note = f"BASE model (no saved LoRA adapter: {type(e).__name__})"
        print(f"WARNING: no saved LoRA adapter ({type(e).__name__}), using the BASE model. Run Task 4 first.")

# Daily trading log: everything printed below also goes to a file on Drive.
import contextlib, sys
log_dir = os.path.join(CACHE_DIR, "trading_logs")
os.makedirs(log_dir, exist_ok=True)
now_utc = datetime.datetime.now(datetime.timezone.utc)
log_path = os.path.join(log_dir, f"{now_utc.date().isoformat()}.log")

class _Tee:
    """Writes to the notebook output and to the log file at the same time."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for s in self.streams:
            s.write(text)
            s.flush()
        return len(text)
    def flush(self):
        for s in self.streams:
            s.flush()

with open(log_path, "a", encoding="utf-8") as log_file, contextlib.redirect_stdout(_Tee(sys.stdout, log_file)):
    print(f"\n===== Run started {now_utc.strftime('%Y-%m-%d %H:%M')} UTC =====")
    print(f"  Model: {model_note}")
    print("\nStarting trading loop...")
    print(f"  Symbols: {len(WHITELIST)} stocks")
    print(f"  Iterations: 80")
    print(f"  Interval: {LOOP_SECONDS}s")
    print("-" * 60)

    try:
        run_trading_loop(tc, tok, trading_model, memory=memory,
                         collector=collector, n_iterations=80)
        print("\n\nTrading loop completed successfully!")
        print("\nData collected:")
        collector.summary()
    except NotImplementedError:
        print("Task 5 not yet implemented")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc(file=sys.stdout)

print(f"\nTrading log saved to {log_path}")


### Optional: Inspect & Edit Memory

Use these cells to view, manually edit, download, or upload `trading_memory.json`.
You can add your own reflections or remove bad ones before the trading loop uses them.


In [ ]:
# =============================================================================
# OPTIONAL: Inspect & Edit trading_memory.json
# Run each sub-block independently as needed.
# =============================================================================

import os, json
from google.colab import files

memory_path = os.path.join(CACHE_DIR, "trading_memory.json")

# -- 1. View all memory entries --------------------------------------------
if os.path.exists(memory_path):
    with open(memory_path) as f:
        memories = json.load(f)
    print(f"Found {len(memories)} memory entries:\n")
    for i, m in enumerate(memories):
        label = m.get('label', '?')
        pnl   = m.get('pnl', '?')
        refl  = m.get('reflection', '')[:120].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      {refl}")
        print()
else:
    print("No trading_memory.json found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(memory_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment the block below after you have edited the file locally:
# uploaded = files.upload()   # select your edited trading_memory.json
# for fname, content in uploaded.items():
#     with open(memory_path, 'wb') as f:
#         f.write(content)
# with open(memory_path) as f:
#     memories = json.load(f)
# print(f"Reloaded {len(memories)} memories from uploaded file.")

# -- 4. Delete a bad entry by index ---------------------------------------
# Example: remove entry at index 2
# BAD_IDX = 2
# if os.path.exists(memory_path):
#     with open(memory_path) as f:
#         memories = json.load(f)
#     removed = memories.pop(BAD_IDX)
#     with open(memory_path, 'w') as f:
#         json.dump(memories, f, indent=2)
#     print(f"Removed entry {BAD_IDX}: {removed.get('reflection','')[:60]}")
#     print(f"{len(memories)} entries remain.")


---
## Optional Challenges

### 1. DPO (Direct Preference Optimization)
Instead of SFT (training only on "good" examples), use **DPO** to train on preference pairs:

```python
from trl import DPOTrainer, DPOConfig

# Format: (prompt, chosen_response, rejected_response)
# "chosen" = decisions labeled "good"
# "rejected" = decisions labeled "bad"

dpo_dataset = Dataset.from_dict({
    "prompt": [e["prompt"] for e in pairs],
    "chosen": [e["good_completion"] for e in pairs],
    "rejected": [e["bad_completion"] for e in pairs],
})

config = DPOConfig(
    output_dir="dpo_output",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=5e-5,
)

trainer = DPOTrainer(
    model=lora_model,
    ref_model=None,        # with a LoRA model, trl uses it with the adapter off as the reference
    train_dataset=dpo_dataset,
    processing_class=tok,
    args=config,
)
trainer.train()
```

**Key difference**: SFT teaches "do this", DPO teaches "prefer this over that" — generally produces better-calibrated models.

### 2. Vector Database RAG
Replace BM25 with semantic search:
- Use sentence embeddings (`sentence-transformers`)
- Store in FAISS vector index
- Retrieve by cosine similarity

### 3. Advanced Trading Strategy
- Position sizing based on confidence scores
- Stop-loss and take-profit logic
- Portfolio performance tracking with matplotlib

### 4. More Data Sources
- SEC filings (EDGAR API)
- Social media sentiment (Reddit, Twitter)
- Options chain data
- Earnings calendar integration